# Accuracy–collocation–runtime tradeoff and loss ablations (Ex 20.2)

**Paper:** *Gaeta–Lie Neural SDEs: Symmetry-Regularized Learning of Stochastic Dynamics* (NeurIPS 2026) — Sections 5.6–5.7, Appendix N.1, Tables 17–19.

Ex 20.2 with 5 or 10 training trajectories: symmetry-batch sizes S ∈ {96, 192, 384, 768} with equal trajectory/jitter/box mixtures, plus one-at-a-time loss-weight sweeps, zero-weight component ablations, library and generator-count ablations. Results are checkpointed and resumable.

**Pipeline naming used in the code:** Stage I = baseline neural SDE surrogate; Stage II = symmetry discovery over the basis library; Stage F = Stage III (symmetry-regularized refinement).

Run the cells top to bottom. All configuration lives in `CONFIG` in the first code cell. See the [repository README](../../README.md) for installation and the full experiment index.

## Cell 1 — Imports, 20D configuration, and E10 experiment specification

In [ ]:
# =========================================================================
# Cell 1 — Imports, 20D configuration, and E10 experiment specification
# =============================================================================

from jax import config as jax_config
jax_config.update("jax_enable_x64", False)

import gc
import json
import math
import os
import pickle
import resource
import time
import zlib
from functools import partial

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

print("JAX version:", jax.__version__)
print("JAX devices:", jax.devices())


CONFIG = {
    "SEED": 20260728,

    # ------------------------------------------------------------------
    # 20D synthetic SDE and simulation.
    # The default is manuscript Ex 20.2, chosen because it is a difficult,
    # nonlinear coupled system on which 5–10 trajectories produce a visibly
    # imperfect vanilla surrogate.
    # ------------------------------------------------------------------
    "STATE_DIM": 20,
    "TRUE_SDE_NAME": "tamed_lorenz_chain_20d",
    "T_FINAL": 2.0,
    "DT": 0.01,
    "MAX_TRAIN_TRAJECTORIES": 10,
    "N_HOLDOUT_TRAJECTORIES": 8,
    "X0_MEAN": 0.0,
    "X0_STD": 0.8,

    # Neural SDE surrogate: manuscript 20D architecture.
    "MLP_HIDDEN_DIM": 96,
    "MLP_DEPTH": 3,
    "INPUT_T_SCALE": 2.0,
    "INPUT_X_SCALE": 3.0,
    "SIGMA_EPS": 1e-4,

    # Stage I: manuscript 20D settings.
    "BASELINE_EPOCHS": 2000,
    "BASELINE_BATCH_SIZE": 512,
    "BASELINE_LR": 2e-3,

    # Stage II: manuscript 20D settings.
    "SYM_DISCOVERY_EPOCHS": 2500,
    "SYM_DISCOVERY_LR": 1e-2,
    "SYM_DISCOVERY_POINTS": 384,  # 128 from each mixture component.
    "SYM_INIT_SCALE": 0.05,
    "LAMBDA_DET": 1.0,
    "LAMBDA_CLOSE": 0.1,
    "LAMBDA_IND": 0.05,
    "LAMBDA_PROJECTABLE": 1.0,
    "LAMBDA_NORM_HINGE": 0.01,
    "MIN_GENERATOR_NORM": 0.10,
    "CLOSURE_RCOND": 1e-5,

    # Full manuscript 20D library uses this rate grid and has R=208.
    "BASIS_LINEAR_MU_LIST": [-2.0, -0.5, 0.5, 2.0],
    "BASIS_EXP_CLIP": 7.0,

    # Stage III: manuscript 20D settings.
    "STAGE_F_EPOCHS": 1200,
    "STAGE_F_BATCH_SIZE": 512,
    "STAGE_F_LR": 1e-3,
    "STAGE_F_LAMBDA_SYM_DET": 0.05,
    "STAGE_F_LAMBDA_PROJECTABLE": 1e-3,

    # Diagnostics.
    "N_COEFFICIENT_EVAL_POINTS": 2000,
    "N_SYMMETRY_HOLDOUT_POINTS": 384,
    "PRINT_EVERY": 250,
    "PLOT_COMPONENTS": [0, 1, 4, 9, 14, 19],
}

D = int(CONFIG["STATE_DIM"])
Q = D + 1

_DEFAULT_OUTPUT_DIR = "./e10_20d_outputs"

E10_CONFIG = {
    # Two requested nested data-scarce regimes.
    "TRAJECTORY_COUNTS": [10, 5],

    # Three seeds for the accuracy–runtime curve.
    "TRADEOFF_SEEDS": [42, 43, 44],

    # Full E10 ablations are substantially more expensive. One diagnostic seed
    # is the default compute-saving choice; add 43 and 44 here if desired.
    "ABLATION_SEEDS": [42],

    # Exact equal mixtures. These totals are divisible by three.
    "COLLOCATION_TOTAL_SWEEP": [96, 192, 384, 768],
    "REFERENCE_COLLOCATION_TOTAL": 384,
    "COLLOCATION_MIX": {"data": 1.0 / 3.0, "jitter": 1.0 / 3.0, "uniform": 1.0 / 3.0},
    "JITTER_STD": 0.05,

    # Reference 20D E10 configuration.
    "REFERENCE_LIBRARY": "full_208",
    "REFERENCE_M": 5,

    # E10 ablations, evaluated at REFERENCE_COLLOCATION_TOTAL.
    "LIBRARY_SWEEP": ["reduced_44", "full_208"],
    "M_SWEEP": [1, 3, 5, 8, 12, 22],
    "OBJECTIVE_MODES": ["full", "no_closure", "no_ind_norm", "det_only"],
    "WEIGHT_MULTIPLIERS": [0.1, 1.0, 10.0],
    "WEIGHTS_TO_SWEEP": [
        "lambda_det", "lambda_close", "lambda_ind", "lambda_norm", "lambda_gl"
    ],

    # Fixed scientific evaluation clouds.
    "SIMULATION_SEED": 20260728,
    "EVAL_SEED": 20260729,
    "GRAM_RANK_RTOL": 1e-6,
    "M_PLATEAU_RELATIVE_TOLERANCE": 0.05,

    # Support-aware box construction from each scarce training set.
    "SUPPORT_QUANTILE_LOW": 0.005,
    "SUPPORT_QUANTILE_HIGH": 0.995,
    "SUPPORT_MARGIN_FACTOR": 1.25,
    "SUPPORT_MIN_HALFWIDTH": 1.0,

    # Resumability.
    "OUTPUT_DIR": _DEFAULT_OUTPUT_DIR,
    "STATE_FILENAME": "E10_20D_resume_state.pkl",
    "RESUME": True,
    "CLEAR_JAX_CACHES_BETWEEN_RUNS": False,

    # Run switches.
    "RUN_COLLOCATION_TRADEOFF": True,
    "RUN_STAGE_CONTROLS": True,
    "RUN_OBJECTIVE_ABLATION": True,
    "RUN_LIBRARY_ABLATION": True,
    "RUN_M_SWEEP": True,
    "RUN_WEIGHT_SWEEP": True,
}

REFERENCE_DISCOVERY_WEIGHTS = {
    "lambda_det": float(CONFIG["LAMBDA_DET"]),
    "lambda_close": float(CONFIG["LAMBDA_CLOSE"]),
    "lambda_ind": float(CONFIG["LAMBDA_IND"]),
    "lambda_norm": float(CONFIG["LAMBDA_NORM_HINGE"]),
    "lambda_projectable": float(CONFIG["LAMBDA_PROJECTABLE"]),
}

print("\nE10 20D protocol")
print("  state dimension d                 :", D)
print("  Kozlov upper bound d+2            :", D + 2)
print("  training-trajectory regimes       :", E10_CONFIG["TRAJECTORY_COUNTS"])
print("  collocation totals                :", E10_CONFIG["COLLOCATION_TOTAL_SWEEP"])
print("  ambient extras (2/3 of totals)    :", [2 * s // 3 for s in E10_CONFIG["COLLOCATION_TOTAL_SWEEP"]])
print("  tradeoff seeds                    :", E10_CONFIG["TRADEOFF_SEEDS"])
print("  ablation seeds                    :", E10_CONFIG["ABLATION_SEEDS"])
print("  reference (R,m,S_total)           : (208, 5, 384)")
print("  output directory                  :", E10_CONFIG["OUTPUT_DIR"])


## Cell 2 — 20D manuscript SDE registry and nested scarce-data simulation

In [ ]:
# =========================================================================
# Cell 2 — 20D manuscript SDE registry and nested scarce-data simulation
# =============================================================================


def _idx_float():
    return jnp.arange(D, dtype=jnp.float32)


def _phase_vector():
    return 2.0 * jnp.pi * _idx_float() / float(D)


# -----------------------------------------------------------------------------
# Ex 20.1: phase-difference ring.
# -----------------------------------------------------------------------------
def drift_phase_difference_ring_20d(t, x):
    d_plus = jnp.roll(x, -1, axis=-1) - x
    d_minus = jnp.roll(x, 1, axis=-1) - x
    d_minus2 = jnp.roll(x, 2, axis=-1) - x
    return (
        0.75 * jnp.sin(d_plus)
        + 0.45 * jnp.sin(d_minus)
        + 0.20 * jnp.sin(2.0 * d_plus)
        + 0.15 * jnp.sin(d_minus2)
    )


def sigma_phase_difference_ring_20d(t, x):
    return jnp.broadcast_to(
        0.28 + 0.04 * jnp.sin(_phase_vector() + 0.3), x.shape
    )


# -----------------------------------------------------------------------------
# Ex 20.2: tamed Lorenz chain — default difficult E10 example.
# -----------------------------------------------------------------------------
def drift_tamed_lorenz_chain_20d(t, x):
    theta = _phase_vector()
    x_plus = jnp.roll(x, -1, axis=-1)
    x_minus = jnp.roll(x, 1, axis=-1)
    numerator = (
        2.8 * (x_plus - x)
        + 0.9 * x_minus * (1.5 - 0.15 * x_plus)
        - 0.85 * x
        + 0.08 * jnp.sin(1.2 * t + theta)
    )
    denominator = 1.0 + 0.04 * jnp.sum(x**2, axis=-1, keepdims=True)
    return numerator / denominator


def sigma_tamed_lorenz_chain_20d(t, x):
    theta = _phase_vector()
    base = 0.18 + 0.025 * jnp.sin(theta + 0.2)
    state_amp = 0.035 + 0.008 * jnp.sin(theta + 1.2)
    return base + state_amp * jnp.tanh(x) ** 2


# -----------------------------------------------------------------------------
# Ex 20.3: cyclic reaction ring.
# -----------------------------------------------------------------------------
def drift_cyclic_reaction_ring_20d(t, x):
    theta = _phase_vector()
    return (
        -(0.50 + 0.10 * jnp.sin(theta)) * x
        + 0.72 * jnp.tanh(jnp.roll(x, -1, axis=-1))
        - 0.30 * jnp.tanh(jnp.roll(x, 1, axis=-1))
        + 0.18 * jnp.tanh(jnp.roll(x, -2, axis=-1))
        + 0.08 * jnp.sin(t + theta)
    )


def sigma_cyclic_reaction_ring_20d(t, x):
    theta = _phase_vector()
    base = 0.24 + 0.020 * jnp.sin(theta + 0.5)
    state_amp = 0.045 + 0.010 * jnp.sin(theta + 1.1)
    return base + state_amp * jnp.sin(x) ** 2


SDE_EXAMPLES = {
    "phase_difference_ring_20d": {
        "description": "Manuscript Ex 20.1: nonlinear phase-difference ring.",
        "drift_fn": drift_phase_difference_ring_20d,
        "sigma_fn": sigma_phase_difference_ring_20d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
    },
    "tamed_lorenz_chain_20d": {
        "description": "Manuscript Ex 20.2: nonlinear tamed Lorenz chain.",
        "drift_fn": drift_tamed_lorenz_chain_20d,
        "sigma_fn": sigma_tamed_lorenz_chain_20d,
        "t_final": 2.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 0.8,
        "input_x_scale": 3.0,
    },
    "cyclic_reaction_ring_20d": {
        "description": "Manuscript Ex 20.3: cyclic nonlinear reaction ring.",
        "drift_fn": drift_cyclic_reaction_ring_20d,
        "sigma_fn": sigma_cyclic_reaction_ring_20d,
        "t_final": 3.0,
        "dt": 0.01,
        "x0_mean": 0.0,
        "x0_std": 1.0,
        "input_x_scale": 3.0,
    },
}


def activate_sde_from_config():
    name = CONFIG["TRUE_SDE_NAME"]
    if name not in SDE_EXAMPLES:
        raise ValueError(f"Unknown TRUE_SDE_NAME={name}; choices={list(SDE_EXAMPLES)}")
    spec = SDE_EXAMPLES[name]
    CONFIG["T_FINAL"] = float(spec["t_final"])
    CONFIG["DT"] = float(spec["dt"])
    CONFIG["X0_MEAN"] = float(spec["x0_mean"])
    CONFIG["X0_STD"] = float(spec["x0_std"])
    CONFIG["INPUT_T_SCALE"] = float(spec["t_final"])
    CONFIG["INPUT_X_SCALE"] = float(spec["input_x_scale"])
    return spec


ACTIVE_SDE = activate_sde_from_config()


def true_drift(t, x):
    return ACTIVE_SDE["drift_fn"](t, x)


def true_sigma(t, x):
    return jnp.maximum(ACTIVE_SDE["sigma_fn"](t, x), 1e-6)


print("\n" + "=" * 84)
print("ACTIVE 20D SYNTHETIC SDE")
print("=" * 84)
print("name       :", CONFIG["TRUE_SDE_NAME"])
print("description:", ACTIVE_SDE["description"])
print("T, dt      :", CONFIG["T_FINAL"], CONFIG["DT"])
print("train pool :", CONFIG["MAX_TRAIN_TRAJECTORIES"], "trajectories")
print("holdout    :", CONFIG["N_HOLDOUT_TRAJECTORIES"], "independent trajectories")
print("=" * 84)


@partial(jax.jit, static_argnums=(1,))
def simulate_sde_paths(key, n_paths):
    T = float(CONFIG["T_FINAL"])
    dt = float(CONFIG["DT"])
    n_steps = int(round(T / dt))
    key_x0, key_noise = jax.random.split(key)
    x0 = (
        float(CONFIG["X0_MEAN"])
        + float(CONFIG["X0_STD"])
        * jax.random.normal(key_x0, (int(n_paths), D))
    )
    noise = jax.random.normal(key_noise, (n_steps, int(n_paths), D))
    times = jnp.arange(n_steps + 1, dtype=jnp.float32) * dt
    sqrt_dt = jnp.sqrt(dt)

    def step(x, inp):
        t, zeta = inp
        x_next = x + true_drift(t, x) * dt + true_sigma(t, x) * sqrt_dt * zeta
        return x_next, x_next

    _, tail = jax.lax.scan(step, x0, (times[:-1], noise))
    paths = jnp.concatenate([x0[None, :, :], tail], axis=0)
    return times, jnp.transpose(paths, (1, 0, 2))


def make_increment_dataset(times, paths):
    n_paths, n_times, dim = paths.shape
    n_steps = n_times - 1
    t0 = jnp.broadcast_to(times[:-1][None, :], (n_paths, n_steps)).reshape(-1)
    x0 = paths[:, :-1, :].reshape(-1, dim)
    dx = (paths[:, 1:, :] - paths[:, :-1, :]).reshape(-1, dim)
    dt_arr = jnp.full((x0.shape[0],), CONFIG["DT"], dtype=jnp.float32)
    return t0, x0, dt_arr, dx


def sample_batch(key, data, batch_size):
    n = data[0].shape[0]
    idx = jax.random.randint(key, (int(batch_size),), 0, n)
    return tuple(arr[idx] for arr in data)


n_total_paths = int(CONFIG["MAX_TRAIN_TRAJECTORIES"] + CONFIG["N_HOLDOUT_TRAJECTORIES"])
sim_key = jax.random.PRNGKey(int(E10_CONFIG["SIMULATION_SEED"]))
times, all_paths = simulate_sde_paths(sim_key, n_total_paths)
jax.block_until_ready(all_paths)

TRAIN_POOL_PATHS = all_paths[: CONFIG["MAX_TRAIN_TRAJECTORIES"]]
HOLDOUT_PATHS = all_paths[CONFIG["MAX_TRAIN_TRAJECTORIES"] :]

# Four independent validation and four independent test trajectories.
_n_holdout = int(HOLDOUT_PATHS.shape[0])
_n_val_paths = _n_holdout // 2
VAL_PATHS = HOLDOUT_PATHS[:_n_val_paths]
TEST_PATHS = HOLDOUT_PATHS[_n_val_paths:]
VAL_DATA = make_increment_dataset(times, VAL_PATHS)
TEST_DATA = make_increment_dataset(times, TEST_PATHS)

print("times shape      :", times.shape)
print("train-pool paths :", TRAIN_POOL_PATHS.shape)
print("validation paths :", VAL_PATHS.shape)
print("test paths       :", TEST_PATHS.shape)

for comp in CONFIG["PLOT_COMPONENTS"]:
    plt.figure(figsize=(8, 3.2))
    for i in range(min(10, TRAIN_POOL_PATHS.shape[0])):
        plt.plot(np.asarray(times), np.asarray(TRAIN_POOL_PATHS[i, :, comp]), alpha=0.55)
    plt.xlabel("t")
    plt.ylabel(f"X_{comp + 1}(t)")
    plt.title(f"{CONFIG['TRUE_SDE_NAME']}: training-pool component {comp + 1}")
    plt.grid(True)
    plt.show()

x_probe = TRAIN_POOL_PATHS[:2, :5, :].reshape(-1, D)
t_probe = 0.5 * CONFIG["T_FINAL"]
print("drift probe shape:", true_drift(t_probe, x_probe).shape)
print("sigma probe shape:", true_sigma(t_probe, x_probe).shape)
print("sigma minimum    :", float(jnp.min(true_sigma(t_probe, x_probe))))


# %% =========================================================================
# Cell 2B — Scarce-regime bundles, equal three-way collocation mixture,
#           fixed evaluation clouds, and utility helpers
# =============================================================================


def stable_name_code(name):
    return np.uint32(zlib.crc32(str(name).encode("utf-8")) & 0xFFFFFFFF)


def named_key(seed, name, step=0):
    k = jax.random.PRNGKey(int(seed))
    k = jax.random.fold_in(k, stable_name_code(name))
    return jax.random.fold_in(k, int(step))


def tree_copy(tree):
    return jax.tree_util.tree_map(lambda x: jnp.array(x, copy=True), tree)


def tree_to_numpy(tree):
    return jax.tree_util.tree_map(lambda x: np.asarray(x), tree)


def tree_to_jax(tree):
    return jax.tree_util.tree_map(lambda x: jnp.asarray(x), tree)


def tree_all_finite(tree):
    return all(np.all(np.isfinite(np.asarray(x))) for x in jax.tree_util.tree_leaves(tree))


def tree_max_abs_diff(a, b):
    aa = jax.tree_util.tree_leaves(a)
    bb = jax.tree_util.tree_leaves(b)
    if len(aa) != len(bb):
        return float("inf")
    return float(max(np.max(np.abs(np.asarray(x) - np.asarray(y))) for x, y in zip(aa, bb)))


def _collocation_counts(total):
    total = int(total)
    if total <= 0 or total % 3 != 0:
        raise ValueError("Every collocation total must be positive and divisible by 3.")
    n = total // 3
    return {"data": n, "jitter": n, "uniform": n, "ambient_extra": 2 * n}


def _support_box_from_training_states(x_train):
    x_np = np.asarray(x_train)
    qlo = np.quantile(x_np, E10_CONFIG["SUPPORT_QUANTILE_LOW"], axis=0)
    qhi = np.quantile(x_np, E10_CONFIG["SUPPORT_QUANTILE_HIGH"], axis=0)
    center = 0.5 * (qlo + qhi)
    half = 0.5 * (qhi - qlo)
    half = np.maximum(
        E10_CONFIG["SUPPORT_MARGIN_FACTOR"] * half,
        E10_CONFIG["SUPPORT_MIN_HALFWIDTH"],
    )
    return jnp.asarray(center - half, dtype=jnp.float32), jnp.asarray(center + half, dtype=jnp.float32)


def sample_tx_data(key, data, n_points):
    t0, x0, _, _ = data
    idx = jax.random.randint(key, (int(n_points),), 0, t0.shape[0])
    return jnp.concatenate([t0[idx, None], x0[idx]], axis=1)


def sample_tx_jitter(key, data, n_points, jitter_std=None):
    if jitter_std is None:
        jitter_std = E10_CONFIG["JITTER_STD"]
    key_idx, key_noise = jax.random.split(key)
    tx = sample_tx_data(key_idx, data, n_points)
    x = tx[:, 1:] + float(jitter_std) * jax.random.normal(key_noise, (int(n_points), D))
    return jnp.concatenate([tx[:, :1], x], axis=1)


def sample_tx_uniform_box(key, n_points, t_box, x_box):
    key_t, key_x = jax.random.split(key)
    t = jax.random.uniform(
        key_t, (int(n_points), 1), minval=float(t_box[0]), maxval=float(t_box[1])
    )
    low, high = x_box
    u = jax.random.uniform(key_x, (int(n_points), D))
    x = low[None, :] + u * (high[None, :] - low[None, :])
    return jnp.concatenate([t, x], axis=1)


def sample_tx_mixture(key, bundle, total):
    counts = _collocation_counts(total)
    k_data, k_jitter, k_uniform, k_perm = jax.random.split(key, 4)
    tx_data = sample_tx_data(k_data, bundle["train"], counts["data"])
    tx_jitter = sample_tx_jitter(k_jitter, bundle["train"], counts["jitter"])
    tx_uniform = sample_tx_uniform_box(
        k_uniform,
        counts["uniform"],
        (0.0, CONFIG["T_FINAL"]),
        bundle["x_box"],
    )
    tx = jnp.concatenate([tx_data, tx_jitter, tx_uniform], axis=0)
    perm = jax.random.permutation(k_perm, tx.shape[0])
    return tx[perm]


def sample_tx_mixture_nested(key, bundle, total):
    """
    Generate the maximum 768-point equal mixture and take equal-component
    prefixes. Thus every smaller budget is nested inside the larger budget for
    the same seed and epoch, improving the fairness of the budget comparison.
    """
    total = int(total)
    max_total = int(max(E10_CONFIG["COLLOCATION_TOTAL_SWEEP"]))
    c = _collocation_counts(total)
    cmax = _collocation_counts(max_total)
    k_data, k_jitter, k_uniform, k_perm = jax.random.split(key, 4)
    data_all = sample_tx_data(k_data, bundle["train"], cmax["data"])
    jitter_all = sample_tx_jitter(k_jitter, bundle["train"], cmax["jitter"])
    uniform_all = sample_tx_uniform_box(
        k_uniform,
        cmax["uniform"],
        (0.0, CONFIG["T_FINAL"]),
        bundle["x_box"],
    )
    tx = jnp.concatenate(
        [data_all[: c["data"]], jitter_all[: c["jitter"]], uniform_all[: c["uniform"]]],
        axis=0,
    )
    perm = jax.random.permutation(k_perm, tx.shape[0])
    return tx[perm]


def _nearest_state_distance_stats(tx, train_x, max_reference=512):
    """RMS-per-coordinate Euclidean distance to a fixed training-state subset."""
    x = np.asarray(tx[:, 1:])
    ref = np.asarray(train_x[: min(int(train_x.shape[0]), int(max_reference))])
    # Small diagnostic clouds only; chunk to avoid a large temporary array.
    mins = []
    for start in range(0, x.shape[0], 64):
        chunk = x[start : start + 64]
        dist2 = np.mean((chunk[:, None, :] - ref[None, :, :]) ** 2, axis=-1)
        mins.append(np.sqrt(np.min(dist2, axis=1)))
    d = np.concatenate(mins)
    return {
        "nearest_train_rms_mean": float(np.mean(d)),
        "nearest_train_rms_median": float(np.median(d)),
        "nearest_train_rms_p90": float(np.quantile(d, 0.90)),
    }


def build_regime_bundle(n_trajectories):
    n_trajectories = int(n_trajectories)
    train_paths = TRAIN_POOL_PATHS[:n_trajectories]
    train = make_increment_dataset(times, train_paths)
    x_box = _support_box_from_training_states(train[1])

    eval_seed = int(E10_CONFIG["EVAL_SEED"])
    tx_coeff = sample_tx_uniform_box(
        named_key(eval_seed, f"coeff_eval:ntraj={n_trajectories}"),
        CONFIG["N_COEFFICIENT_EVAL_POINTS"],
        (0.0, CONFIG["T_FINAL"]),
        x_box,
    )
    coeff_t, coeff_x = tx_coeff[:, 0], tx_coeff[:, 1:]
    f_true = jax.vmap(lambda tt, xx: true_drift(tt, xx))(coeff_t, coeff_x)
    sigma_true = jax.vmap(lambda tt, xx: true_sigma(tt, xx))(coeff_t, coeff_x)

    bundle = {
        "n_trajectories": n_trajectories,
        "train_paths": train_paths,
        "train": train,
        "val": VAL_DATA,
        "test": TEST_DATA,
        "x_box": x_box,
        "tx_coeff": tx_coeff,
        "coeff_f_true": f_true,
        "coeff_sigma_true": sigma_true,
    }

    bundle["tx_sym_val"] = sample_tx_mixture(
        named_key(eval_seed, f"sym_val:ntraj={n_trajectories}"),
        bundle,
        CONFIG["N_SYMMETRY_HOLDOUT_POINTS"],
    )
    bundle["tx_sym_test"] = sample_tx_mixture(
        named_key(eval_seed, f"sym_test:ntraj={n_trajectories}"),
        bundle,
        CONFIG["N_SYMMETRY_HOLDOUT_POINTS"],
    )

    distance_by_budget = {}
    for total in E10_CONFIG["COLLOCATION_TOTAL_SWEEP"]:
        tx_diag = sample_tx_mixture_nested(
            named_key(eval_seed, f"distance_diag:ntraj={n_trajectories}:S={total}"),
            bundle,
            total,
        )
        distance_by_budget[int(total)] = _nearest_state_distance_stats(tx_diag, train[1])
    bundle["distance_by_budget"] = distance_by_budget
    return bundle


E10_DATA_BUNDLES = {
    int(n): build_regime_bundle(n) for n in E10_CONFIG["TRAJECTORY_COUNTS"]
}

for n, bundle in E10_DATA_BUNDLES.items():
    low, high = bundle["x_box"]
    print("\nScarce regime:", n, "training trajectories")
    print("  training transitions:", bundle["train"][0].shape[0])
    print("  validation transitions:", bundle["val"][0].shape[0])
    print("  test transitions:", bundle["test"][0].shape[0])
    print("  mean box width:", float(jnp.mean(high - low)))
    for total in E10_CONFIG["COLLOCATION_TOTAL_SWEEP"]:
        c = _collocation_counts(total)
        dstat = bundle["distance_by_budget"][int(total)]
        print(
            f"  S={total:4d}: data/jitter/uniform={c['data']}/{c['jitter']}/{c['uniform']}, "
            f"ambient={c['ambient_extra']}, nearest-RMS mean={dstat['nearest_train_rms_mean']:.4f}"
        )

## Cell 3 — Reusable 20D neural SDE, Adam, Stage-I training, and evaluation

In [ ]:
# =========================================================================
# Cell 3 — Reusable 20D neural SDE, Adam, Stage-I training, and evaluation
# =============================================================================


def glorot_uniform(key, in_dim, out_dim):
    lim = jnp.sqrt(6.0 / (in_dim + out_dim))
    return jax.random.uniform(key, (in_dim, out_dim), minval=-lim, maxval=lim)


class MLP:
    def __init__(self, in_dim, hidden_dim, depth, out_dim):
        self.in_dim = int(in_dim)
        self.hidden_dim = int(hidden_dim)
        self.depth = int(depth)
        self.out_dim = int(out_dim)

    def init(self, key):
        sizes = [self.in_dim] + [self.hidden_dim] * self.depth + [self.out_dim]
        keys = jax.random.split(key, len(sizes) - 1)
        params = []
        for k, din, dout in zip(keys, sizes[:-1], sizes[1:]):
            params.append({
                "W": glorot_uniform(k, din, dout),
                "b": jnp.zeros((dout,), dtype=jnp.float32),
            })
        return params

    def apply(self, params, z):
        h = z
        for layer in params[:-1]:
            h = jnp.tanh(h @ layer["W"] + layer["b"])
        return h @ params[-1]["W"] + params[-1]["b"]


surrogate_model = MLP(
    in_dim=Q,
    hidden_dim=CONFIG["MLP_HIDDEN_DIM"],
    depth=CONFIG["MLP_DEPTH"],
    out_dim=2 * D,
)


def normalize_input(t, x):
    return jnp.concatenate([
        jnp.asarray([t / CONFIG["INPUT_T_SCALE"]], dtype=jnp.float32),
        x / CONFIG["INPUT_X_SCALE"],
    ])


def surrogate_f_sigma(params, t, x):
    raw = surrogate_model.apply(params, normalize_input(t, x))
    f = raw[:D]
    sigma = jax.nn.softplus(raw[D:]) + CONFIG["SIGMA_EPS"]
    return f, sigma


def f_vec_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1:])[0]


def sigma_vec_single(params, z):
    return surrogate_f_sigma(params, z[0], z[1:])[1]


def data_nll_loss(params, batch):
    t0, x0, dt, dx = batch
    f, sigma = jax.vmap(lambda tt, xx: surrogate_f_sigma(params, tt, xx))(t0, x0)
    mean = f * dt[:, None]
    var = sigma**2 * dt[:, None] + 1e-8
    terms = 0.5 * ((dx - mean) ** 2 / var + jnp.log(2.0 * jnp.pi * var))
    return jnp.mean(jnp.sum(terms, axis=-1))


def adam_init(params):
    return {
        "m": jax.tree_util.tree_map(jnp.zeros_like, params),
        "v": jax.tree_util.tree_map(jnp.zeros_like, params),
        "t": jnp.array(0, dtype=jnp.int32),
    }


def adam_update(params, grads, state, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    t_new = state["t"] + 1
    m_new = jax.tree_util.tree_map(
        lambda m, g: beta1 * m + (1.0 - beta1) * g,
        state["m"], grads,
    )
    v_new = jax.tree_util.tree_map(
        lambda v, g: beta2 * v + (1.0 - beta2) * g * g,
        state["v"], grads,
    )
    m_hat = jax.tree_util.tree_map(lambda m: m / (1.0 - beta1**t_new), m_new)
    v_hat = jax.tree_util.tree_map(lambda v: v / (1.0 - beta2**t_new), v_new)
    params_new = jax.tree_util.tree_map(
        lambda p, m, v: p - float(lr) * m / (jnp.sqrt(v) + eps),
        params, m_hat, v_hat,
    )
    return params_new, {"m": m_new, "v": v_new, "t": t_new}


def coefficient_metrics(params, bundle):
    tx = bundle["tx_coeff"]
    f_pred, sigma_pred = jax.vmap(
        lambda tt, xx: surrogate_f_sigma(params, tt, xx)
    )(tx[:, 0], tx[:, 1:])
    return {
        "drift_mse": float(jnp.mean((f_pred - bundle["coeff_f_true"]) ** 2)),
        "sigma_mse": float(jnp.mean((sigma_pred - bundle["coeff_sigma_true"]) ** 2)),
    }


def evaluate_surrogate(params, bundle):
    return {
        "train_nll": float(data_nll_loss(params, bundle["train"])),
        "val_nll": float(data_nll_loss(params, bundle["val"])),
        "test_nll": float(data_nll_loss(params, bundle["test"])),
        **coefficient_metrics(params, bundle),
    }


def make_data_train_step(lr):
    @jax.jit
    def step(params, opt_state, batch):
        loss, grads = jax.value_and_grad(data_nll_loss)(params, batch)
        params, opt_state = adam_update(params, grads, opt_state, lr)
        return params, opt_state, loss
    return step


def train_stage1(seed, bundle):
    ntraj = int(bundle["n_trajectories"])
    params = surrogate_model.init(named_key(seed, f"stage1_init:ntraj={ntraj}"))
    opt_state = adam_init(params)
    step = make_data_train_step(CONFIG["BASELINE_LR"])
    history = []

    start = time.perf_counter()
    for epoch in range(1, CONFIG["BASELINE_EPOCHS"] + 1):
        batch = sample_batch(
            named_key(seed, f"stage1_batch:ntraj={ntraj}", epoch),
            bundle["train"],
            CONFIG["BASELINE_BATCH_SIZE"],
        )
        params, opt_state, loss = step(params, opt_state, batch)
        if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
            jax.block_until_ready(loss)
            row = {"epoch": epoch, "batch_nll": float(loss)}
            history.append(row)
            print(
                f"[Stage I | ntraj={ntraj} | seed={seed}] epoch {epoch:5d} | "
                f"batch NLL={float(loss): .6f}"
            )

    jax.block_until_ready(jax.tree_util.tree_leaves(params)[0])
    elapsed = time.perf_counter() - start
    metrics = evaluate_surrogate(params, bundle)

    if not tree_all_finite(params) or not all(np.isfinite(v) for v in metrics.values()):
        raise FloatingPointError(f"Non-finite Stage-I result: ntraj={ntraj}, seed={seed}")

    print(
        f"[Stage I | ntraj={ntraj} | seed={seed}] complete | "
        f"val NLL={metrics['val_nll']:.6f} | drift MSE={metrics['drift_mse']:.4e} | "
        f"sigma MSE={metrics['sigma_mse']:.4e} | time={elapsed:.1f}s"
    )
    return {"params": params, "metrics": metrics, "time_s": float(elapsed), "history": history}


## Cell 4 — Exact reduced/full 20D basis-library factory (R=44 / R=208)

In [ ]:
# =========================================================================
# Cell 4 — Exact reduced/full 20D basis-library factory (R=44 / R=208)
# =============================================================================


E10_LIBRARY_CACHE = {}


def build_basis_library(mode):
    """
    reduced_44:
        ∂t (1), ∂xi (20), lifted Brownian scaling (1),
        t∂t and t²∂t (2), xi∂xi (20). Total R=44.

    full_208:
        reduced_44 plus exponential translations (80), lifted projective
        terms (4), nearest-neighbor differences (20), rotations (20), and
        two oriented nearest-neighbor shear families (40). Total R=208.
    """
    if mode in E10_LIBRARY_CACHE:
        return E10_LIBRARY_CACHE[mode]
    if mode not in {"reduced_44", "full_208"}:
        raise ValueError(f"Unknown 20D library mode: {mode}")

    entries = []

    def add(name, source, fn):
        entries.append({"name": name, "source": source, "fn": fn})

    def zxi():
        return jnp.zeros((D,), dtype=jnp.float32)

    def ei(i):
        return jnp.zeros((D,), dtype=jnp.float32).at[i].set(1.0)

    def safe_exp(v):
        c = float(CONFIG["BASIS_EXP_CLIP"])
        return jnp.exp(jnp.clip(v, -c, c))

    # ∂t.
    add(
        "time_translation__dt",
        "time translation",
        lambda t, x: jnp.concatenate([jnp.array([1.0], dtype=jnp.float32), zxi()]),
    )

    # ∂xi.
    for i in range(D):
        add(
            f"translation__dx{i+1}",
            "component translation",
            lambda t, x, i=i: jnp.concatenate([
                jnp.array([0.0], dtype=jnp.float32), ei(i)
            ]),
        )

    # 2t∂t + x·∇x.
    add(
        "global_brownian_scaling__2t_dt_plus_x_dot_dx",
        "lifted Brownian/Gaeta scaling",
        lambda t, x: jnp.concatenate([
            jnp.array([2.0 * t], dtype=jnp.float32), x
        ]),
    )

    # Time-only distractors.
    add(
        "time_scaling__t_dt",
        "time-only distractor",
        lambda t, x: jnp.concatenate([jnp.array([t], dtype=jnp.float32), zxi()]),
    )
    add(
        "time_quadratic__t2_dt",
        "time-only distractor",
        lambda t, x: jnp.concatenate([jnp.array([t * t], dtype=jnp.float32), zxi()]),
    )

    # xi∂xi.
    for i in range(D):
        add(
            f"scaling__x{i+1}_dx{i+1}",
            "component scaling",
            lambda t, x, i=i: jnp.concatenate([
                jnp.array([0.0], dtype=jnp.float32),
                jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[i]),
            ]),
        )

    if mode == "full_208":
        # e^{mu t} ∂xi and e^{2mu t}(∂t + mu x·∇x).
        for mu_value in CONFIG["BASIS_LINEAR_MU_LIST"]:
            mu = float(mu_value)
            for i in range(D):
                add(
                    f"exp_translation__mu_{mu:+.2f}__dx{i+1}",
                    "component exponential translation",
                    lambda t, x, mu=mu, i=i: jnp.concatenate([
                        jnp.array([0.0], dtype=jnp.float32),
                        jnp.zeros((D,), dtype=jnp.float32).at[i].set(safe_exp(mu * t)),
                    ]),
                )
            add(
                f"exp_projective__mu_{mu:+.2f}",
                "lifted projective exponential generator",
                lambda t, x, mu=mu: jnp.concatenate([
                    jnp.array([safe_exp(2.0 * mu * t)], dtype=jnp.float32),
                    mu * safe_exp(2.0 * mu * t) * x,
                ]),
            )

        # Nearest-neighbor ring differences and rotations.
        for i in range(D):
            ip = (i + 1) % D
            im = (i - 1) % D
            add(
                f"ring_difference__dx{i+1}_minus_dx{ip+1}",
                "nearest-neighbor difference translation",
                lambda t, x, i=i, ip=ip: jnp.concatenate([
                    jnp.array([0.0], dtype=jnp.float32), ei(i) - ei(ip)
                ]),
            )
            add(
                f"ring_rotation__x{ip+1}_dx{i+1}_minus_x{i+1}_dx{ip+1}",
                "nearest-neighbor rotation",
                lambda t, x, i=i, ip=ip: jnp.concatenate([
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[ip]).at[ip].set(-x[i]),
                ]),
            )
            add(
                f"ring_shear_forward__x{ip+1}_dx{i+1}",
                "nearest-neighbor forward shear",
                lambda t, x, i=i, ip=ip: jnp.concatenate([
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[ip]),
                ]),
            )
            add(
                f"ring_shear_backward__x{im+1}_dx{i+1}",
                "nearest-neighbor backward shear",
                lambda t, x, i=i, im=im: jnp.concatenate([
                    jnp.array([0.0], dtype=jnp.float32),
                    jnp.zeros((D,), dtype=jnp.float32).at[i].set(x[im]),
                ]),
            )

    names = tuple(e["name"] for e in entries)
    sources = tuple(e["source"] for e in entries)
    fns = tuple(e["fn"] for e in entries)
    expected = 44 if mode == "reduced_44" else 208
    if len(fns) != expected:
        raise RuntimeError(f"Library {mode} has R={len(fns)}, expected {expected}")

    library = {"mode": mode, "R": len(fns), "names": names, "sources": sources, "fns": fns}
    E10_LIBRARY_CACHE[mode] = library
    print(f"Built 20D library {mode}: R={library['R']}")
    return library


def eval_basis_one(library, t, x):
    return jnp.stack([fn(t, x) for fn in library["fns"]], axis=0)


def eval_generators_point(library, W, z):
    return W @ eval_basis_one(library, z[0], z[1:])


def eval_generators_batch(library, W, tx):
    return jax.vmap(lambda z: eval_generators_point(library, W, z))(tx)


def gram_matrix(library, W, tx):
    X = eval_generators_batch(library, W, tx)
    m = X.shape[1]
    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    return (V.T @ V) / (Q * tx.shape[0])


def whiten_random_W(library, W_raw, tx_reference, eps=1e-5):
    G = gram_matrix(library, W_raw, tx_reference)
    eigvals, eigvecs = jnp.linalg.eigh(G)
    inv_sqrt = eigvecs @ jnp.diag(1.0 / jnp.sqrt(jnp.maximum(eigvals, eps))) @ eigvecs.T
    return inv_sqrt @ W_raw


def make_random_control_W(seed, bundle, library, m):
    ntraj = int(bundle["n_trajectories"])
    raw = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
        named_key(seed, f"random_W:ntraj={ntraj}:lib={library['mode']}:m={m}"),
        (int(m), int(library["R"])),
    )
    return whiten_random_W(library, raw, bundle["tx_sym_val"])


REDUCED_LIBRARY = build_basis_library("reduced_44")
FULL_LIBRARY = build_basis_library("full_208")

## Cell 5 — Itô determining residuals and reusable Stage-II discovery

In [ ]:
# =========================================================================
# Cell 5 — Itô determining residuals and reusable Stage-II discovery
# =============================================================================


def det_residuals_point(sde_params, library, W, z):
    """Projectable Itô determining residuals for diagonal 20D diffusion."""
    f = f_vec_single(sde_params, z)
    sigma = sigma_vec_single(sde_params, z)

    J_f = jax.jacfwd(lambda zz: f_vec_single(sde_params, zz))(z)
    J_sigma = jax.jacfwd(lambda zz: sigma_vec_single(sde_params, zz))(z)
    f_t, f_x = J_f[:, 0], J_f[:, 1:]
    sigma_t, sigma_x = J_sigma[:, 0], J_sigma[:, 1:]

    generator_fn = lambda zz: eval_generators_point(library, W, zz)
    X = generator_fn(z)
    J_X = jax.jacfwd(generator_fn)(z)

    tau = X[:, 0]
    xi = X[:, 1:]
    tau_t = J_X[:, 0, 0]
    tau_x = J_X[:, 0, 1:]
    xi_t = J_X[:, 1:, 0]
    xi_x = J_X[:, 1:, 1:]

    # All templates in the 20D library are at most linear in x, so the
    # Itô Hessian term (1/2) a^{jk} ∂_{x_jx_k} xi is exactly zero.
    adv_xi = jnp.einsum("j,aij->ai", f, xi_x)
    xi_grad_f = jnp.einsum("aj,ij->ai", xi, f_x)
    R_f = (
        xi_t
        + adv_xi
        - xi_grad_f
        - tau[:, None] * f_t[None, :]
        - tau_t[:, None] * f[None, :]
    )

    # For diagonal Sigma, Sigma^{j alpha}=delta_{j alpha} sigma_alpha.
    eye = jnp.eye(D, dtype=jnp.float32)
    first = jnp.einsum("p,aip->aip", sigma, xi_x)
    diagonal_term = (
        jnp.einsum("aj,ij->ai", xi, sigma_x)
        + tau[:, None] * sigma_t[None, :]
        + 0.5 * tau_t[:, None] * sigma[None, :]
    )
    R_sigma = first - diagonal_term[:, :, None] * eye[None, :, :]
    return R_f, R_sigma, tau_x


def determining_parts(sde_params, library, W, tx):
    R_f, R_sigma, tau_x = jax.vmap(
        lambda z: det_residuals_point(sde_params, library, W, z)
    )(tx)
    return jnp.mean(R_f**2), jnp.mean(R_sigma**2), jnp.mean(tau_x**2)


def generator_values_and_jacobians(library, W, tx):
    def one(z):
        fn = lambda zz: eval_generators_point(library, W, zz)
        return fn(z), jax.jacfwd(fn)(z)
    return jax.vmap(one)(tx)


def lie_closure_loss_global(library, W, tx, closure_rcond):
    m = int(W.shape[0])
    if m < 2:
        return jnp.array(0.0, dtype=jnp.float32)

    X, J = generator_values_and_jacobians(library, W, tx)
    ia, ib = jnp.triu_indices(m, k=1)
    Xa, Xb = X[:, ia, :], X[:, ib, :]
    Ja, Jb = J[:, ia, :, :], J[:, ib, :, :]
    brackets = (
        jnp.einsum("spc,spqc->spq", Xa, Jb)
        - jnp.einsum("spc,spqc->spq", Xb, Ja)
    )

    V = jnp.transpose(X, (1, 0, 2)).reshape(m, -1).T
    B = jnp.transpose(brackets, (1, 0, 2)).reshape(brackets.shape[1], -1).T
    G = V.T @ V
    reg = (float(closure_rcond) ** 2) * jnp.eye(m, dtype=G.dtype)
    C = jnp.linalg.solve(G + reg, V.T @ B)
    return jnp.mean((B - V @ C) ** 2)


def gram_losses(library, W, tx, min_generator_norm):
    G = gram_matrix(library, W, tx)
    I = jnp.eye(W.shape[0], dtype=G.dtype)
    independence = jnp.mean((G - I) ** 2)
    norm_hinge = jnp.mean(
        jnp.maximum(0.0, float(min_generator_norm) - jnp.diag(G)) ** 2
    )
    return independence, norm_hinge, G


def discovery_weights_for_mode(mode):
    w = dict(REFERENCE_DISCOVERY_WEIGHTS)
    if mode == "full":
        pass
    elif mode == "no_closure":
        w["lambda_close"] = 0.0
    elif mode == "no_ind_norm":
        w["lambda_ind"] = 0.0
        w["lambda_norm"] = 0.0
    elif mode == "det_only":
        w["lambda_close"] = 0.0
        w["lambda_ind"] = 0.0
        w["lambda_norm"] = 0.0
    else:
        raise ValueError(f"Unknown discovery objective mode: {mode}")
    return w


def discovery_weight_tuple(weights):
    return tuple(float(weights[k]) for k in (
        "lambda_det", "lambda_close", "lambda_ind",
        "lambda_norm", "lambda_projectable",
    ))


def make_symmetry_train_step(library, weights):
    l_det, l_close, l_ind, l_norm, l_proj = discovery_weight_tuple(weights)

    def objective(W, sde_params, tx):
        drift_det, sigma_det, projectable = determining_parts(
            sde_params, library, W, tx
        )
        close = (
            lie_closure_loss_global(library, W, tx, CONFIG["CLOSURE_RCOND"])
            if l_close != 0.0 else jnp.array(0.0, dtype=jnp.float32)
        )
        if l_ind != 0.0 or l_norm != 0.0:
            ind, norm_hinge, _ = gram_losses(
                library, W, tx, CONFIG["MIN_GENERATOR_NORM"]
            )
        else:
            ind = jnp.array(0.0, dtype=jnp.float32)
            norm_hinge = jnp.array(0.0, dtype=jnp.float32)
        det = drift_det + sigma_det
        total = (
            l_det * det
            + l_close * close
            + l_ind * ind
            + l_norm * norm_hinge
            + l_proj * projectable
        )
        return total, (drift_det, sigma_det, close, ind, norm_hinge, projectable)

    value_and_grad = jax.value_and_grad(objective, argnums=0, has_aux=True)

    @jax.jit
    def step(W, opt_state, sde_params, tx):
        (loss, aux), grads = value_and_grad(W, sde_params, tx)
        W, opt_state = adam_update(W, grads, opt_state, CONFIG["SYM_DISCOVERY_LR"])
        return W, opt_state, loss, aux

    return step


def symmetry_diagnostics(sde_params, library, W, tx):
    drift_det, sigma_det, projectable = determining_parts(sde_params, library, W, tx)
    close = lie_closure_loss_global(library, W, tx, CONFIG["CLOSURE_RCOND"])
    ind, norm_hinge, G = gram_losses(library, W, tx, CONFIG["MIN_GENERATOR_NORM"])

    G_np = np.asarray(G)
    eigvals = np.linalg.eigvalsh(0.5 * (G_np + G_np.T))
    max_eig = float(np.max(eigvals)) if eigvals.size else 0.0
    threshold = E10_CONFIG["GRAM_RANK_RTOL"] * max(max_eig, 1e-12)
    positive = eigvals[eigvals > threshold]
    effective_rank = int(positive.size)
    min_eig = float(np.min(eigvals)) if eigvals.size else np.nan
    condition = float(np.max(positive) / np.min(positive)) if positive.size else float("inf")

    return {
        "stage2_drift_det": float(drift_det),
        "stage2_sigma_det": float(sigma_det),
        "stage2_det": float(drift_det + sigma_det),
        "stage2_close": float(close),
        "stage2_ind": float(ind),
        "stage2_norm": float(norm_hinge),
        "stage2_proj": float(projectable),
        "gram_min_eig": min_eig,
        "gram_eff_rank": effective_rank,
        "gram_condition": condition,
    }


def train_symmetry_discovery(stage1_params, bundle, library, m, weights, seed, display_name):
    ntraj = int(bundle["n_trajectories"])
    frozen_before = tree_copy(stage1_params)
    W = CONFIG["SYM_INIT_SCALE"] * jax.random.normal(
        named_key(seed, f"stage2_init:ntraj={ntraj}:lib={library['mode']}:m={m}"),
        (int(m), int(library["R"])),
    )
    opt_state = adam_init(W)
    step = make_symmetry_train_step(library, weights)
    history = []
    step_times = []

    start = time.perf_counter()
    for epoch in range(1, CONFIG["SYM_DISCOVERY_EPOCHS"] + 1):
        tx = sample_tx_mixture(
            named_key(seed, f"stage2_collocation:ntraj={ntraj}", epoch),
            bundle,
            CONFIG["SYM_DISCOVERY_POINTS"],
        )
        t0 = time.perf_counter()
        W, opt_state, loss, aux = step(W, opt_state, stage1_params, tx)
        jax.block_until_ready(loss)
        step_times.append(time.perf_counter() - t0)

        if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
            drift_det, sigma_det, close, ind, norm_hinge, proj = map(float, aux)
            history.append({
                "epoch": epoch,
                "total": float(loss),
                "drift_det": drift_det,
                "sigma_det": sigma_det,
                "close": close,
                "ind": ind,
                "norm": norm_hinge,
                "projectable": proj,
            })
            print(
                f"[Stage II | ntraj={ntraj} | seed={seed} | {display_name}] "
                f"epoch {epoch:5d} | total={float(loss):.4e} | "
                f"det={drift_det + sigma_det:.4e} | close={close:.3e} | ind={ind:.3e}"
            )

    elapsed = time.perf_counter() - start
    diagnostics = symmetry_diagnostics(stage1_params, library, W, bundle["tx_sym_test"])
    freeze_delta = tree_max_abs_diff(frozen_before, stage1_params)
    steady = np.asarray(step_times[min(3, len(step_times)) :], dtype=float)

    if not np.all(np.isfinite(np.asarray(W))):
        raise FloatingPointError(f"Non-finite Stage-II W: {display_name}")
    if freeze_delta > 1e-10:
        raise RuntimeError(f"Stage II modified theta: max delta={freeze_delta:.3e}")

    diagnostics.update({
        "stage2_time_s": float(elapsed),
        "stage2_mean_step_s": float(np.mean(steady)) if steady.size else np.nan,
        "stage2_p95_step_s": float(np.quantile(steady, 0.95)) if steady.size else np.nan,
        "stage2_param_delta_max": float(freeze_delta),
    })
    print(
        f"[Stage II | ntraj={ntraj} | seed={seed} | {display_name}] complete | "
        f"held-out det={diagnostics['stage2_det']:.4e} | "
        f"rank={diagnostics['gram_eff_rank']}/{m} | time={elapsed:.1f}s"
    )
    return {"W": W, "diagnostics": diagnostics, "history": history}

## Cell 6 — Stage-III refinement, equal-mixture budget control, runtime/memory

In [ ]:
# =========================================================================
# Cell 6 — Stage-III refinement, equal-mixture budget control, runtime/memory
# =============================================================================


def _process_peak_rss_mb():
    value = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
    # Linux reports KiB; macOS reports bytes. Colab/Linux is the main target.
    return value / 1024.0 if value < 1e9 else value / (1024.0**2)


def _device_memory_snapshot():
    try:
        stats = jax.devices()[0].memory_stats() or {}
    except Exception:
        stats = {}
    def get(*names):
        for name in names:
            if name in stats and stats[name] is not None:
                return float(stats[name])
        return np.nan
    return {
        "device_bytes_in_use": get("bytes_in_use", "bytes_in_use.current"),
        "device_peak_bytes": get("peak_bytes_in_use", "bytes_in_use.peak"),
        "device_bytes_limit": get("bytes_limit", "bytes_reservable_limit"),
    }


def make_stage3_symmetry_step(library, lambda_gl):
    lambda_gl = float(lambda_gl)
    lambda_proj = float(CONFIG["STAGE_F_LAMBDA_PROJECTABLE"])

    def objective(params, W_fixed, batch, tx_sym):
        data_loss = data_nll_loss(params, batch)
        drift_det, sigma_det, projectable = determining_parts(
            params, library, W_fixed, tx_sym
        )
        total = (
            data_loss
            + lambda_gl * (drift_det + sigma_det)
            + lambda_proj * projectable
        )
        return total, (data_loss, drift_det, sigma_det, projectable)

    value_and_grad = jax.value_and_grad(objective, argnums=0, has_aux=True)

    @jax.jit
    def step(params, opt_state, W_fixed, batch, tx_sym):
        (loss, aux), grads = value_and_grad(params, W_fixed, batch, tx_sym)
        params, opt_state = adam_update(params, grads, opt_state, CONFIG["STAGE_F_LR"])
        return params, opt_state, loss, aux

    return step


def stage3_residual_diagnostics(params, library, W, tx):
    drift_det, sigma_det, projectable = determining_parts(params, library, W, tx)
    return {
        "stage3_drift_det": float(drift_det),
        "stage3_sigma_det": float(sigma_det),
        "stage3_det": float(drift_det + sigma_det),
        "stage3_proj": float(projectable),
    }


def train_stage3(stage1_params, W_fixed, bundle, library, lambda_gl, seed, collocation_total, display_name):
    ntraj = int(bundle["n_trajectories"])
    collocation_total = int(collocation_total)
    counts = _collocation_counts(collocation_total)
    params = tree_copy(stage1_params)
    opt_state = adam_init(params)
    step = make_stage3_symmetry_step(library, lambda_gl)
    history = []
    step_times = []

    rss_before = _process_peak_rss_mb()
    device_before = _device_memory_snapshot()
    start = time.perf_counter()

    for epoch in range(1, CONFIG["STAGE_F_EPOCHS"] + 1):
        batch = sample_batch(
            named_key(seed, f"stage3_data_batch:ntraj={ntraj}", epoch),
            bundle["train"],
            CONFIG["STAGE_F_BATCH_SIZE"],
        )
        tx_sym = sample_tx_mixture(
            named_key(seed, f"stage3_collocation:ntraj={ntraj}", epoch),
            bundle,
            collocation_total,
        )
        t0 = time.perf_counter()
        params, opt_state, loss, aux = step(params, opt_state, W_fixed, batch, tx_sym)
        jax.block_until_ready(loss)
        step_times.append(time.perf_counter() - t0)

        if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
            data_loss, drift_det, sigma_det, proj = map(float, aux)
            history.append({
                "epoch": epoch,
                "total": float(loss),
                "data": data_loss,
                "drift_det": drift_det,
                "sigma_det": sigma_det,
                "projectable": proj,
            })
            print(
                f"[Stage III | ntraj={ntraj} | seed={seed} | S={collocation_total} | {display_name}] "
                f"epoch {epoch:5d} | total={float(loss):.5f} | data={data_loss:.5f} | "
                f"det={drift_det + sigma_det:.4e}"
            )

    elapsed = time.perf_counter() - start
    metrics = evaluate_surrogate(params, bundle)
    residuals = stage3_residual_diagnostics(params, library, W_fixed, bundle["tx_sym_test"])
    rss_after = _process_peak_rss_mb()
    device_after = _device_memory_snapshot()
    steady = np.asarray(step_times[min(3, len(step_times)) :], dtype=float)

    if not tree_all_finite(params):
        raise FloatingPointError(f"Non-finite Stage-III params: {display_name}")
    if not all(np.isfinite(v) for v in {**metrics, **residuals}.values()):
        raise FloatingPointError(f"Non-finite Stage-III metrics: {display_name}")

    timing = {
        "stage3_time_s": float(elapsed),
        "stage3_mean_step_s": float(np.mean(steady)) if steady.size else np.nan,
        "stage3_median_step_s": float(np.median(steady)) if steady.size else np.nan,
        "stage3_p95_step_s": float(np.quantile(steady, 0.95)) if steady.size else np.nan,
        "stage3_compile_inclusive_mean_step_s": float(np.mean(step_times)),
        "stage3_collocation_points_processed": int(CONFIG["STAGE_F_EPOCHS"] * collocation_total),
        "process_peak_rss_before_mb": float(rss_before),
        "process_peak_rss_after_mb": float(rss_after),
        "process_peak_rss_delta_mb": float(max(0.0, rss_after - rss_before)),
        "device_bytes_in_use_before": device_before["device_bytes_in_use"],
        "device_bytes_in_use_after": device_after["device_bytes_in_use"],
        "device_peak_bytes": device_after["device_peak_bytes"],
    }

    print(
        f"[Stage III | ntraj={ntraj} | seed={seed} | S={collocation_total}] complete | "
        f"val NLL={metrics['val_nll']:.6f} | drift MSE={metrics['drift_mse']:.4e} | "
        f"sigma MSE={metrics['sigma_mse']:.4e} | time={elapsed:.1f}s | "
        f"steady step={timing['stage3_mean_step_s']:.4f}s"
    )
    return {
        "params": params,
        "metrics": metrics,
        "residuals": residuals,
        "timing": timing,
        "history": history,
        "counts": counts,
    }


def train_stage3_data_only(stage1_params, bundle, seed):
    """Same Stage-III data updates and LR, but no symmetry/collocation loss."""
    ntraj = int(bundle["n_trajectories"])
    params = tree_copy(stage1_params)
    opt_state = adam_init(params)
    step = make_data_train_step(CONFIG["STAGE_F_LR"])
    step_times = []

    start = time.perf_counter()
    for epoch in range(1, CONFIG["STAGE_F_EPOCHS"] + 1):
        batch = sample_batch(
            named_key(seed, f"stage3_data_batch:ntraj={ntraj}", epoch),
            bundle["train"],
            CONFIG["STAGE_F_BATCH_SIZE"],
        )
        t0 = time.perf_counter()
        params, opt_state, loss = step(params, opt_state, batch)
        jax.block_until_ready(loss)
        step_times.append(time.perf_counter() - t0)
        if epoch == 1 or epoch % CONFIG["PRINT_EVERY"] == 0:
            print(
                f"[Stage III data-only | ntraj={ntraj} | seed={seed}] epoch {epoch:5d} | "
                f"batch NLL={float(loss):.6f}"
            )

    elapsed = time.perf_counter() - start
    metrics = evaluate_surrogate(params, bundle)
    steady = np.asarray(step_times[min(3, len(step_times)) :], dtype=float)
    if not tree_all_finite(params) or not all(np.isfinite(v) for v in metrics.values()):
        raise FloatingPointError(f"Non-finite data-only Stage III: ntraj={ntraj}, seed={seed}")

    return {
        "params": params,
        "metrics": metrics,
        "timing": {
            "stage3_time_s": float(elapsed),
            "stage3_mean_step_s": float(np.mean(steady)) if steady.size else np.nan,
            "stage3_median_step_s": float(np.median(steady)) if steady.size else np.nan,
            "stage3_p95_step_s": float(np.quantile(steady, 0.95)) if steady.size else np.nan,
            "stage3_compile_inclusive_mean_step_s": float(np.mean(step_times)),
            "stage3_collocation_points_processed": 0,
        },
    }

## Cell 7 — Cell 7 — Resumable experiment runner and flat numerical result records

In [ ]:

# =========================================================================
# Cell 7 — Cell 7 — Resumable experiment runner and flat numerical result records
# =============================================================================


STATE_PROTOCOL = "E10_20D_SCARCE_MIXTURE_V1"

REQUIRED_RESULT_FIELDS = [
    "run_id", "status", "family", "variant", "label", "seed",
    "n_trajectories", "n_train_transitions",
    "library", "R", "m", "objective_mode", "weight_name", "weight_multiplier",
    "lambda_det", "lambda_close", "lambda_ind", "lambda_norm", "lambda_gl",
    "collocation_total", "collocation_data", "collocation_jitter",
    "collocation_uniform", "ambient_extra_points",
    "nearest_train_rms_mean", "nearest_train_rms_median", "nearest_train_rms_p90",
    "train_nll", "val_nll", "test_nll", "drift_mse", "sigma_mse",
    "stage2_det", "stage2_drift_det", "stage2_sigma_det", "stage2_close",
    "stage2_ind", "stage2_norm", "stage2_proj", "gram_min_eig",
    "gram_eff_rank", "gram_condition", "stage2_param_delta_max",
    "stage3_det", "stage3_drift_det", "stage3_sigma_det", "stage3_proj",
    "stage1_time_s", "stage2_time_s", "stage3_time_s", "total_time_s",
    "stage2_mean_step_s", "stage2_p95_step_s",
    "stage3_mean_step_s", "stage3_median_step_s", "stage3_p95_step_s",
    "stage3_compile_inclusive_mean_step_s", "stage3_collocation_points_processed",
    "process_peak_rss_before_mb", "process_peak_rss_after_mb",
    "process_peak_rss_delta_mb", "device_bytes_in_use_before",
    "device_bytes_in_use_after", "device_peak_bytes",
]


def nan_stage2_fields():
    return {
        "stage2_det": np.nan,
        "stage2_drift_det": np.nan,
        "stage2_sigma_det": np.nan,
        "stage2_close": np.nan,
        "stage2_ind": np.nan,
        "stage2_norm": np.nan,
        "stage2_proj": np.nan,
        "gram_min_eig": np.nan,
        "gram_eff_rank": np.nan,
        "gram_condition": np.nan,
        "stage2_param_delta_max": 0.0,
        "stage2_time_s": 0.0,
        "stage2_mean_step_s": np.nan,
        "stage2_p95_step_s": np.nan,
    }


def nan_stage3_fields():
    return {
        "stage3_det": np.nan,
        "stage3_drift_det": np.nan,
        "stage3_sigma_det": np.nan,
        "stage3_proj": np.nan,
        "stage3_time_s": 0.0,
        "stage3_mean_step_s": np.nan,
        "stage3_median_step_s": np.nan,
        "stage3_p95_step_s": np.nan,
        "stage3_compile_inclusive_mean_step_s": np.nan,
        "stage3_collocation_points_processed": 0,
        "process_peak_rss_before_mb": np.nan,
        "process_peak_rss_after_mb": np.nan,
        "process_peak_rss_delta_mb": np.nan,
        "device_bytes_in_use_before": np.nan,
        "device_bytes_in_use_after": np.nan,
        "device_peak_bytes": np.nan,
    }


def weights_key(weights):
    names = ("lambda_det", "lambda_close", "lambda_ind", "lambda_norm", "lambda_projectable")
    return "|".join(f"{name}={float(weights[name]):.12g}" for name in names)


def stage1_cache_key(ntraj, seed):
    return f"ntraj={int(ntraj)}|seed={int(seed)}"


def stage2_cache_key(ntraj, seed, library_mode, m, weights):
    return (
        f"ntraj={int(ntraj)}|seed={int(seed)}|lib={library_mode}|m={int(m)}|"
        f"{weights_key(weights)}"
    )


def condition_cache_key(ntraj, seed, kind, library_mode, m, weights, lambda_gl, collocation_total):
    return (
        f"ntraj={int(ntraj)}|seed={int(seed)}|kind={kind}|lib={library_mode}|m={int(m)}|"
        f"{weights_key(weights)}|lambda_gl={float(lambda_gl):.12g}|S={int(collocation_total)}"
    )


def default_state():
    return {
        "protocol": STATE_PROTOCOL,
        "results": [],
        "stage1": {},
        "stage2": {},
        "conditions": {},
        "data_only": {},
    }


def state_path():
    os.makedirs(E10_CONFIG["OUTPUT_DIR"], exist_ok=True)
    return os.path.join(E10_CONFIG["OUTPUT_DIR"], E10_CONFIG["STATE_FILENAME"])


def load_state():
    path = state_path()
    if E10_CONFIG["RESUME"] and os.path.isfile(path):
        with open(path, "rb") as f:
            state = pickle.load(f)
        if state.get("protocol") != STATE_PROTOCOL:
            print("Ignoring incompatible resume state:", path)
            return default_state()
        for name, value in default_state().items():
            state.setdefault(name, value)
        print("Loaded resume state:", path)
        return state
    return default_state()


def save_state(state):
    path = state_path()
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        pickle.dump(state, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)


def result_exists(state, run_id):
    return any(row.get("run_id") == run_id for row in state["results"])


def append_record(state, record):
    missing = [name for name in REQUIRED_RESULT_FIELDS if name not in record]
    if missing:
        raise RuntimeError(f"Record {record.get('run_id')} missing fields: {missing}")
    if not result_exists(state, record["run_id"]):
        state["results"].append(record)
        save_state(state)


def maybe_clear_jax_caches():
    gc.collect()
    if E10_CONFIG["CLEAR_JAX_CACHES_BETWEEN_RUNS"]:
        jax.clear_caches()


def base_record(
    bundle, seed, run_id, family, variant, label,
    library, m, objective_mode, weight_name, weight_multiplier,
    weights, lambda_gl, collocation_total,
):
    ntraj = int(bundle["n_trajectories"])
    collocation_total = int(collocation_total)
    counts = _collocation_counts(collocation_total) if collocation_total > 0 else {
        "data": 0, "jitter": 0, "uniform": 0, "ambient_extra": 0
    }
    distance = bundle["distance_by_budget"].get(collocation_total, {
        "nearest_train_rms_mean": np.nan,
        "nearest_train_rms_median": np.nan,
        "nearest_train_rms_p90": np.nan,
    })
    rec = {
        "run_id": str(run_id),
        "status": "ok",
        "family": str(family),
        "variant": str(variant),
        "label": str(label),
        "seed": int(seed),
        "n_trajectories": ntraj,
        "n_train_transitions": int(bundle["train"][0].shape[0]),
        "library": library["mode"] if library is not None else "none",
        "R": int(library["R"]) if library is not None else 0,
        "m": int(m),
        "objective_mode": str(objective_mode),
        "weight_name": str(weight_name),
        "weight_multiplier": float(weight_multiplier),
        "lambda_det": float(weights["lambda_det"]),
        "lambda_close": float(weights["lambda_close"]),
        "lambda_ind": float(weights["lambda_ind"]),
        "lambda_norm": float(weights["lambda_norm"]),
        "lambda_gl": float(lambda_gl),
        "collocation_total": collocation_total,
        "collocation_data": int(counts["data"]),
        "collocation_jitter": int(counts["jitter"]),
        "collocation_uniform": int(counts["uniform"]),
        "ambient_extra_points": int(counts["ambient_extra"]),
        **distance,
        "train_nll": np.nan,
        "val_nll": np.nan,
        "test_nll": np.nan,
        "drift_mse": np.nan,
        "sigma_mse": np.nan,
        "stage1_time_s": 0.0,
        "total_time_s": 0.0,
    }
    rec.update(nan_stage2_fields())
    rec.update(nan_stage3_fields())
    return rec


def ensure_stage1(state, bundle, seed):
    ntraj = int(bundle["n_trajectories"])
    key = stage1_cache_key(ntraj, seed)
    if key not in state["stage1"]:
        trained = train_stage1(seed, bundle)
        state["stage1"][key] = {
            "params": tree_to_numpy(trained["params"]),
            "metrics": trained["metrics"],
            "time_s": trained["time_s"],
        }
        save_state(state)
        maybe_clear_jax_caches()
    cached = state["stage1"][key]
    return {
        "params": tree_to_jax(cached["params"]),
        "metrics": dict(cached["metrics"]),
        "time_s": float(cached["time_s"]),
    }


def ensure_stage2(state, bundle, seed, stage1, library, m, weights, display_name):
    ntraj = int(bundle["n_trajectories"])
    key = stage2_cache_key(ntraj, seed, library["mode"], m, weights)
    if key not in state["stage2"]:
        trained = train_symmetry_discovery(
            stage1["params"], bundle, library, m, weights, seed, display_name
        )
        state["stage2"][key] = {
            "W": np.asarray(trained["W"]),
            "diagnostics": trained["diagnostics"],
        }
        save_state(state)
        maybe_clear_jax_caches()
    cached = state["stage2"][key]
    return {"W": jnp.asarray(cached["W"]), "diagnostics": dict(cached["diagnostics"])}


def ensure_data_only(state, bundle, seed, stage1):
    ntraj = int(bundle["n_trajectories"])
    key = f"ntraj={ntraj}|seed={int(seed)}"
    if key not in state["data_only"]:
        trained = train_stage3_data_only(stage1["params"], bundle, seed)
        state["data_only"][key] = {
            **trained["metrics"],
            **trained["timing"],
            **nan_stage2_fields(),
            **{k: v for k, v in nan_stage3_fields().items() if k not in trained["timing"]},
            "total_time_s": float(trained["timing"]["stage3_time_s"]),
        }
        save_state(state)
        maybe_clear_jax_caches()
    return dict(state["data_only"][key])


def ensure_symmetry_condition(
    state, bundle, seed, stage1, library, m, weights, lambda_gl,
    collocation_total, display_name, kind="learned",
):
    ntraj = int(bundle["n_trajectories"])
    ckey = condition_cache_key(
        ntraj, seed, kind, library["mode"], m, weights, lambda_gl, collocation_total
    )
    if ckey not in state["conditions"]:
        if kind == "learned":
            stage2 = ensure_stage2(
                state, bundle, seed, stage1, library, m, weights, display_name
            )
            W = stage2["W"]
            stage2_diag = stage2["diagnostics"]
        elif kind == "random":
            W = make_random_control_W(seed, bundle, library, m)
            stage2_diag = symmetry_diagnostics(
                stage1["params"], library, W, bundle["tx_sym_test"]
            )
            stage2_diag.update({
                "stage2_time_s": 0.0,
                "stage2_mean_step_s": np.nan,
                "stage2_p95_step_s": np.nan,
                "stage2_param_delta_max": 0.0,
            })
        else:
            raise ValueError(f"Unknown Stage-III condition kind: {kind}")

        stage3 = train_stage3(
            stage1["params"], W, bundle, library, lambda_gl, seed,
            collocation_total, display_name,
        )
        core = {
            **stage3["metrics"],
            **stage2_diag,
            **stage3["residuals"],
            **stage3["timing"],
            "total_time_s": float(stage2_diag["stage2_time_s"] + stage3["timing"]["stage3_time_s"]),
        }
        state["conditions"][ckey] = core
        save_state(state)
        maybe_clear_jax_caches()
    return dict(state["conditions"][ckey])


def add_view(
    state, bundle, seed, run_id, family, variant, label,
    library, m, objective_mode, weight_name, weight_multiplier,
    weights, lambda_gl, collocation_total, core,
):
    if result_exists(state, run_id):
        return
    rec = base_record(
        bundle, seed, run_id, family, variant, label,
        library, m, objective_mode, weight_name, weight_multiplier,
        weights, lambda_gl, collocation_total,
    )
    rec.update(core)
    append_record(state, rec)


def add_stage1_view(state, bundle, seed, stage1):
    ntraj = int(bundle["n_trajectories"])
    run_id = f"ntraj{ntraj}__seed{seed}__stage1"
    if result_exists(state, run_id):
        return
    rec = base_record(
        bundle, seed, run_id, "stage", "stage1_baseline", "Stage I: vanilla surrogate",
        None, 0, "none", "none", 1.0,
        REFERENCE_DISCOVERY_WEIGHTS, 0.0, 0,
    )
    rec.update(stage1["metrics"])
    rec["stage1_time_s"] = float(stage1["time_s"])
    rec["total_time_s"] = float(stage1["time_s"])
    append_record(state, rec)


def add_stage2_frozen_view(state, bundle, seed, stage1, stage2, library, m, weights):
    ntraj = int(bundle["n_trajectories"])
    run_id = f"ntraj{ntraj}__seed{seed}__stage2_frozen"
    if result_exists(state, run_id):
        return
    rec = base_record(
        bundle, seed, run_id, "stage", "stage2_frozen",
        "Stage II: learned W; theta and coefficient MSE frozen",
        library, m, "full", "none", 1.0, weights,
        CONFIG["STAGE_F_LAMBDA_SYM_DET"], 0,
    )
    rec.update(stage1["metrics"])
    rec.update(stage2["diagnostics"])
    rec["stage1_time_s"] = float(stage1["time_s"])
    rec["total_time_s"] = float(stage2["diagnostics"]["stage2_time_s"])
    append_record(state, rec)


def run_e10_20d():
    state = load_state()
    full_library = build_basis_library("full_208")
    reduced_library = build_basis_library("reduced_44")
    ref_weights = dict(REFERENCE_DISCOVERY_WEIGHTS)
    ref_m = int(E10_CONFIG["REFERENCE_M"])
    ref_gl = float(CONFIG["STAGE_F_LAMBDA_SYM_DET"])
    ref_S = int(E10_CONFIG["REFERENCE_COLLOCATION_TOTAL"])

    all_needed_seeds = sorted(set(E10_CONFIG["TRADEOFF_SEEDS"]) | set(E10_CONFIG["ABLATION_SEEDS"]))

    for ntraj in E10_CONFIG["TRAJECTORY_COUNTS"]:
        bundle = E10_DATA_BUNDLES[int(ntraj)]
        print("\n" + "#" * 116)
        print(f"E10 DATA-SCARCE REGIME: {ntraj} TRAINING TRAJECTORIES")
        print("#" * 116)

        for seed in all_needed_seeds:
            seed = int(seed)
            print("\n" + "-" * 116)
            print(f"ntraj={ntraj} | seed={seed}")
            print("-" * 116)

            stage1 = ensure_stage1(state, bundle, seed)
            add_stage1_view(state, bundle, seed, stage1)

            # Data-only continuation is the compute/further-training control.
            data_only = ensure_data_only(state, bundle, seed, stage1)
            add_view(
                state, bundle, seed,
                f"ntraj{ntraj}__seed{seed}__stage3_data_only",
                "stage", "stage3_data_only", "Stage III: data-only continuation",
                None, 0, "none", "none", 1.0, ref_weights, 0.0, 0, data_only,
            )

            # Reference Stage-II checkpoint is reused across all four budgets.
            ref_stage2 = ensure_stage2(
                state, bundle, seed, stage1, full_library, ref_m, ref_weights,
                "reference full 20D objective",
            )

            if seed in E10_CONFIG["ABLATION_SEEDS"]:
                add_stage2_frozen_view(
                    state, bundle, seed, stage1, ref_stage2, full_library, ref_m, ref_weights
                )

            # --------------------------------------------------------------
            # Accuracy / additional-collocation / runtime curve.
            # --------------------------------------------------------------
            if E10_CONFIG["RUN_COLLOCATION_TRADEOFF"] and seed in E10_CONFIG["TRADEOFF_SEEDS"]:
                for S in E10_CONFIG["COLLOCATION_TOTAL_SWEEP"]:
                    core = ensure_symmetry_condition(
                        state, bundle, seed, stage1, full_library, ref_m,
                        ref_weights, ref_gl, int(S), f"collocation tradeoff S={S}",
                        kind="learned",
                    )
                    add_view(
                        state, bundle, seed,
                        f"ntraj{ntraj}__seed{seed}__tradeoff__S{int(S)}",
                        "collocation_tradeoff", f"S_{int(S)}",
                        f"Full Gaeta-Lie, S_total={int(S)}",
                        full_library, ref_m, "full", "none", 1.0,
                        ref_weights, ref_gl, int(S), core,
                    )

            if seed not in E10_CONFIG["ABLATION_SEEDS"]:
                continue

            # The reference Stage-III condition is already cached if ref_S was
            # part of the tradeoff sweep.
            ref_core = ensure_symmetry_condition(
                state, bundle, seed, stage1, full_library, ref_m,
                ref_weights, ref_gl, ref_S, "full reference", kind="learned",
            )
            add_view(
                state, bundle, seed,
                f"ntraj{ntraj}__seed{seed}__stage3_full",
                "stage", "stage3_full", "Stage III: full learned-W Gaeta-Lie",
                full_library, ref_m, "full", "none", 1.0,
                ref_weights, ref_gl, ref_S, ref_core,
            )

            # Random-W stage control.
            if E10_CONFIG["RUN_STAGE_CONTROLS"]:
                random_core = ensure_symmetry_condition(
                    state, bundle, seed, stage1, full_library, ref_m,
                    ref_weights, ref_gl, ref_S, "Gram-normalized random W",
                    kind="random",
                )
                add_view(
                    state, bundle, seed,
                    f"ntraj{ntraj}__seed{seed}__stage3_random_W",
                    "stage", "stage3_random_W", "Stage III: Gram-normalized random W",
                    full_library, ref_m, "random", "none", 1.0,
                    ref_weights, ref_gl, ref_S, random_core,
                )

            # Discovery-objective ablation.
            if E10_CONFIG["RUN_OBJECTIVE_ABLATION"]:
                labels = {
                    "full": "Full discovery objective",
                    "no_closure": "No closure loss",
                    "no_ind_norm": "No independence or norm control",
                    "det_only": "Determining residual only",
                }
                for mode in E10_CONFIG["OBJECTIVE_MODES"]:
                    mode_weights = discovery_weights_for_mode(mode)
                    core = ref_core if mode == "full" else ensure_symmetry_condition(
                        state, bundle, seed, stage1, full_library, ref_m,
                        mode_weights, ref_gl, ref_S, f"objective={mode}", kind="learned",
                    )
                    add_view(
                        state, bundle, seed,
                        f"ntraj{ntraj}__seed{seed}__objective__{mode}",
                        "objective", f"objective_{mode}", labels[mode],
                        full_library, ref_m, mode, "none", 1.0,
                        mode_weights, ref_gl, ref_S, core,
                    )

            # Reduced/full library ablation.
            if E10_CONFIG["RUN_LIBRARY_ABLATION"]:
                for library in (reduced_library, full_library):
                    core = ref_core if library["mode"] == "full_208" else ensure_symmetry_condition(
                        state, bundle, seed, stage1, library, ref_m,
                        ref_weights, ref_gl, ref_S, f"library={library['mode']}", kind="learned",
                    )
                    add_view(
                        state, bundle, seed,
                        f"ntraj{ntraj}__seed{seed}__library__{library['mode']}",
                        "library", f"library_{library['mode']}",
                        f"{library['mode']} (R={library['R']})",
                        library, ref_m, "full", "none", 1.0,
                        ref_weights, ref_gl, ref_S, core,
                    )

            # Number of learned generators m.
            if E10_CONFIG["RUN_M_SWEEP"]:
                for m in E10_CONFIG["M_SWEEP"]:
                    m = int(m)
                    core = ref_core if m == ref_m else ensure_symmetry_condition(
                        state, bundle, seed, stage1, full_library, m,
                        ref_weights, ref_gl, ref_S, f"m={m}", kind="learned",
                    )
                    add_view(
                        state, bundle, seed,
                        f"ntraj{ntraj}__seed{seed}__m__{m}",
                        "m_sweep", f"m_{m}", f"m={m}",
                        full_library, m, "full", "none", 1.0,
                        ref_weights, ref_gl, ref_S, core,
                    )

            # One-at-a-time 0.1x / 1x / 10x weight sensitivity.
            if E10_CONFIG["RUN_WEIGHT_SWEEP"]:
                for weight_name in E10_CONFIG["WEIGHTS_TO_SWEEP"]:
                    for multiplier in E10_CONFIG["WEIGHT_MULTIPLIERS"]:
                        multiplier = float(multiplier)
                        varied_weights = dict(ref_weights)
                        varied_gl = ref_gl
                        if weight_name == "lambda_gl":
                            varied_gl = ref_gl * multiplier
                        else:
                            varied_weights[weight_name] = ref_weights[weight_name] * multiplier

                        core = ref_core if abs(multiplier - 1.0) < 1e-12 else ensure_symmetry_condition(
                            state, bundle, seed, stage1, full_library, ref_m,
                            varied_weights, varied_gl, ref_S,
                            f"{weight_name}={multiplier:g}x", kind="learned",
                        )
                        add_view(
                            state, bundle, seed,
                            f"ntraj{ntraj}__seed{seed}__weight__{weight_name}__x{multiplier:g}",
                            "weight_sweep", f"{weight_name}_x{multiplier:g}",
                            f"{weight_name}: {multiplier:g}x",
                            full_library, ref_m, "full", weight_name, multiplier,
                            varied_weights, varied_gl, ref_S, core,
                        )

            print(f"Completed/loaded requested rows for ntraj={ntraj}, seed={seed}.")

    metadata = {
        "protocol": STATE_PROTOCOL,
        "sde_name": CONFIG["TRUE_SDE_NAME"],
        "sde_description": ACTIVE_SDE["description"],
        "d": D,
        "kozlov_bound": D + 2,
        "trajectory_counts": list(map(int, E10_CONFIG["TRAJECTORY_COUNTS"])),
        "tradeoff_seeds": list(map(int, E10_CONFIG["TRADEOFF_SEEDS"])),
        "ablation_seeds": list(map(int, E10_CONFIG["ABLATION_SEEDS"])),
        "collocation_total_sweep": list(map(int, E10_CONFIG["COLLOCATION_TOTAL_SWEEP"])),
        "ambient_extra_sweep": [2 * int(s) // 3 for s in E10_CONFIG["COLLOCATION_TOTAL_SWEEP"]],
        "collocation_mix": dict(E10_CONFIG["COLLOCATION_MIX"]),
        "reference_collocation_total": int(ref_S),
        "reference_library": E10_CONFIG["REFERENCE_LIBRARY"],
        "reference_R": int(full_library["R"]),
        "reference_m": int(ref_m),
        "stage1_epochs": int(CONFIG["BASELINE_EPOCHS"]),
        "stage2_epochs": int(CONFIG["SYM_DISCOVERY_EPOCHS"]),
        "stage2_collocation_total": int(CONFIG["SYM_DISCOVERY_POINTS"]),
        "stage3_epochs": int(CONFIG["STAGE_F_EPOCHS"]),
        "validation_description": (
            f"{VAL_PATHS.shape[0]} independent held-out trajectories; "
            f"test uses {TEST_PATHS.shape[0]} separate independent trajectories"
        ),
        "coefficient_eval_description": (
            f"fixed {CONFIG['N_COEFFICIENT_EVAL_POINTS']}-point box-uniform cloud per trajectory regime"
        ),
        "output_dir": E10_CONFIG["OUTPUT_DIR"],
    }
    return state["results"], metadata


E10_RESULTS, E10_METADATA = run_e10_20d()
print("\nE10 20D runner complete.")
print("Flat result rows:", len(E10_RESULTS))
print("Resume-state file:", state_path())

## Final Cell — Numerical report, accuracy–runtime tradeoff, E10 ablations, and CSV/JSON exports

In [ ]:
# =========================================================================
# Final Cell — Numerical report, accuracy–runtime tradeoff,
#              E10 ablations, and CSV/JSON exports
# =============================================================================

import json
import os

import numpy as np
import pandas as pd


if "E10_RESULTS" not in globals() or not isinstance(E10_RESULTS, list) or not E10_RESULTS:
    raise RuntimeError("E10_RESULTS is missing or empty. Run Cell 7 first.")
if "E10_METADATA" not in globals() or not isinstance(E10_METADATA, dict):
    raise RuntimeError("E10_METADATA is missing. Run Cell 7 first.")

os.makedirs(E10_CONFIG["OUTPUT_DIR"], exist_ok=True)
raw = pd.DataFrame(E10_RESULTS).copy()
missing = [name for name in REQUIRED_RESULT_FIELDS if name not in raw.columns]
if missing:
    raise RuntimeError("E10 result records are missing: " + ", ".join(missing))

failed = raw[raw["status"] != "ok"].copy()
ok = raw[raw["status"] == "ok"].copy()
if not failed.empty:
    print("WARNING: failed rows detected")
    print(failed[["run_id", "status"]].to_string(index=False))

# -----------------------------------------------------------------------------
# Attach paired Stage-I and data-only references to every row.
# -----------------------------------------------------------------------------
baseline = (
    ok[ok["variant"] == "stage1_baseline"]
    [["n_trajectories", "seed", "drift_mse", "sigma_mse", "val_nll", "test_nll", "stage1_time_s"]]
    .drop_duplicates(["n_trajectories", "seed"])
    .rename(columns={
        "drift_mse": "baseline_drift_mse",
        "sigma_mse": "baseline_sigma_mse",
        "val_nll": "baseline_val_nll",
        "test_nll": "baseline_test_nll",
        "stage1_time_s": "baseline_stage1_time_s",
    })
)

data_only = (
    ok[ok["variant"] == "stage3_data_only"]
    [["n_trajectories", "seed", "drift_mse", "sigma_mse", "val_nll", "test_nll", "stage3_time_s", "stage3_mean_step_s"]]
    .drop_duplicates(["n_trajectories", "seed"])
    .rename(columns={
        "drift_mse": "dataonly_drift_mse",
        "sigma_mse": "dataonly_sigma_mse",
        "val_nll": "dataonly_val_nll",
        "test_nll": "dataonly_test_nll",
        "stage3_time_s": "dataonly_stage3_time_s",
        "stage3_mean_step_s": "dataonly_mean_step_s",
    })
)

enriched = ok.merge(baseline, on=["n_trajectories", "seed"], how="left")
enriched = enriched.merge(data_only, on=["n_trajectories", "seed"], how="left")

for metric in ("drift", "sigma"):
    enriched[f"{metric}_reduction_vs_stage1_pct"] = 100.0 * (
        1.0 - enriched[f"{metric}_mse"] / enriched[f"baseline_{metric}_mse"]
    )
    enriched[f"{metric}_reduction_vs_dataonly_pct"] = 100.0 * (
        1.0 - enriched[f"{metric}_mse"] / enriched[f"dataonly_{metric}_mse"]
    )

enriched["combined_relative_error_vs_stage1"] = 0.5 * (
    enriched["drift_mse"] / enriched["baseline_drift_mse"]
    + enriched["sigma_mse"] / enriched["baseline_sigma_mse"]
)
enriched["combined_reduction_vs_stage1_pct"] = 100.0 * (
    1.0 - enriched["combined_relative_error_vs_stage1"]
)
enriched["dataonly_combined_relative_error_vs_stage1"] = 0.5 * (
    enriched["dataonly_drift_mse"] / enriched["baseline_drift_mse"]
    + enriched["dataonly_sigma_mse"] / enriched["baseline_sigma_mse"]
)
enriched["combined_gain_over_dataonly_pct_points"] = 100.0 * (
    enriched["dataonly_combined_relative_error_vs_stage1"]
    - enriched["combined_relative_error_vs_stage1"]
)
enriched["extra_stage3_time_over_dataonly_s"] = (
    enriched["stage3_time_s"] - enriched["dataonly_stage3_time_s"]
)
enriched["stage3_runtime_multiplier_vs_dataonly"] = (
    enriched["stage3_time_s"] / enriched["dataonly_stage3_time_s"]
)
enriched["gain_over_dataonly_pct_points_per_extra_second"] = (
    enriched["combined_gain_over_dataonly_pct_points"]
    / enriched["extra_stage3_time_over_dataonly_s"].clip(lower=1e-9)
)
enriched["device_peak_mb"] = enriched["device_peak_bytes"] / (1024.0**2)


def mean_std_frame(df, group_cols, value_cols):
    rows = []
    for keys, group in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))
        row["n_seeds"] = int(group["seed"].nunique())
        for col in value_cols:
            values = pd.to_numeric(group[col], errors="coerce").dropna().to_numpy(dtype=float)
            row[f"{col}_mean"] = float(np.mean(values)) if values.size else np.nan
            row[f"{col}_std"] = float(np.std(values, ddof=1)) if values.size > 1 else 0.0 if values.size == 1 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def pm(mean, std, scientific=False, digits=3):
    if not np.isfinite(mean):
        return "—"
    if scientific:
        return f"{mean:.3e} ± {std:.2e}"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


def print_section(title):
    print("\n" + "=" * 128)
    print(title)
    print("=" * 128)


print_section("FINAL E10 20D PROTOCOL")
print("SDE                            :", E10_METADATA["sde_name"])
print("Description                    :", E10_METADATA["sde_description"])
print("State dimension d              :", E10_METADATA["d"])
print("Kozlov upper bound d+2         :", E10_METADATA["kozlov_bound"])
print("Training trajectories          :", E10_METADATA["trajectory_counts"])
print("Collocation totals             :", E10_METADATA["collocation_total_sweep"])
print("Ambient extras (jitter+uniform):", E10_METADATA["ambient_extra_sweep"])
print("Mixture                        : 1/3 data + 1/3 jitter + 1/3 box-uniform")
print("Tradeoff seeds                 :", E10_METADATA["tradeoff_seeds"])
print("Ablation seeds                 :", E10_METADATA["ablation_seeds"])
print("Reference library / R / m      :", E10_METADATA["reference_library"], E10_METADATA["reference_R"], E10_METADATA["reference_m"])
print("Validation/test protocol       :", E10_METADATA["validation_description"])
print("Coefficient evaluation         :", E10_METADATA["coefficient_eval_description"])


# -----------------------------------------------------------------------------
# Stage-I quality: directly verifies that the chosen 5/10-trajectory settings
# are genuinely data-scarce and reports the exact vanilla errors.
# -----------------------------------------------------------------------------
print_section("A. STAGE-I VANILLA SURROGATE QUALITY IN THE TWO DATA-SCARCE REGIMES")
stage1_summary = mean_std_frame(
    enriched[enriched["variant"] == "stage1_baseline"],
    ["n_trajectories"],
    ["drift_mse", "sigma_mse", "val_nll", "test_nll", "stage1_time_s"],
).sort_values("n_trajectories", ascending=False)

stage1_display = pd.DataFrame({
    "Training trajectories": stage1_summary["n_trajectories"],
    "Seeds": stage1_summary["n_seeds"],
    "Vanilla drift MSE": [pm(a, b, True) for a, b in zip(stage1_summary["drift_mse_mean"], stage1_summary["drift_mse_std"])],
    "Vanilla diffusion MSE": [pm(a, b, True) for a, b in zip(stage1_summary["sigma_mse_mean"], stage1_summary["sigma_mse_std"])],
    "Validation NLL": [pm(a, b) for a, b in zip(stage1_summary["val_nll_mean"], stage1_summary["val_nll_std"])],
    "Test NLL": [pm(a, b) for a, b in zip(stage1_summary["test_nll_mean"], stage1_summary["test_nll_std"])],
    "Stage-I time (s)": [pm(a, b, digits=1) for a, b in zip(stage1_summary["stage1_time_s_mean"], stage1_summary["stage1_time_s_std"])],
})
print(stage1_display.to_string(index=False))


# -----------------------------------------------------------------------------
# Main accuracy–collocation–runtime tradeoff.
# -----------------------------------------------------------------------------
print_section("B. COLLOCATION BUDGET VERSUS COEFFICIENT ACCURACY AND STAGE-III COST")
tradeoff = enriched[enriched["family"] == "collocation_tradeoff"].copy()
tradeoff_values = [
    "drift_mse", "sigma_mse", "val_nll", "test_nll",
    "drift_reduction_vs_stage1_pct", "sigma_reduction_vs_stage1_pct",
    "combined_reduction_vs_stage1_pct", "combined_gain_over_dataonly_pct_points",
    "stage2_time_s", "stage3_time_s", "total_time_s",
    "extra_stage3_time_over_dataonly_s", "stage3_mean_step_s", "stage3_p95_step_s",
    "stage3_runtime_multiplier_vs_dataonly",
    "gain_over_dataonly_pct_points_per_extra_second",
    "process_peak_rss_delta_mb", "device_peak_mb",
    "nearest_train_rms_mean",
]
tradeoff_summary = mean_std_frame(
    tradeoff,
    ["n_trajectories", "collocation_total", "collocation_data", "collocation_jitter", "collocation_uniform", "ambient_extra_points"],
    tradeoff_values,
).sort_values(["n_trajectories", "collocation_total"], ascending=[False, True])

tradeoff_display = pd.DataFrame({
    "Ntraj": tradeoff_summary["n_trajectories"],
    "S total": tradeoff_summary["collocation_total"],
    "data/jitter/box": [
        f"{int(a)}/{int(b)}/{int(c)}" for a, b, c in zip(
            tradeoff_summary["collocation_data"], tradeoff_summary["collocation_jitter"], tradeoff_summary["collocation_uniform"]
        )
    ],
    "ambient extras": tradeoff_summary["ambient_extra_points"],
    "Drift MSE": [pm(a, b, True) for a, b in zip(tradeoff_summary["drift_mse_mean"], tradeoff_summary["drift_mse_std"])],
    "Diffusion MSE": [pm(a, b, True) for a, b in zip(tradeoff_summary["sigma_mse_mean"], tradeoff_summary["sigma_mse_std"])],
    "Drift gain vs vanilla (%)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["drift_reduction_vs_stage1_pct_mean"], tradeoff_summary["drift_reduction_vs_stage1_pct_std"])],
    "Diffusion gain vs vanilla (%)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["sigma_reduction_vs_stage1_pct_mean"], tradeoff_summary["sigma_reduction_vs_stage1_pct_std"])],
    "Combined gain (%)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["combined_reduction_vs_stage1_pct_mean"], tradeoff_summary["combined_reduction_vs_stage1_pct_std"])],
    "Test NLL": [pm(a, b) for a, b in zip(tradeoff_summary["test_nll_mean"], tradeoff_summary["test_nll_std"])],
    "Stage-II time (s)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["stage2_time_s_mean"], tradeoff_summary["stage2_time_s_std"])],
    "Stage-III time (s)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["stage3_time_s_mean"], tradeoff_summary["stage3_time_s_std"])],
    "Stage-II+III time (s)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["total_time_s_mean"], tradeoff_summary["total_time_s_std"])],
    "extra Stage-III time vs data-only (s)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["extra_stage3_time_over_dataonly_s_mean"], tradeoff_summary["extra_stage3_time_over_dataonly_s_std"])],
    "steady step (s)": [pm(a, b, digits=4) for a, b in zip(tradeoff_summary["stage3_mean_step_s_mean"], tradeoff_summary["stage3_mean_step_s_std"])],
    "p95 step (s)": [pm(a, b, digits=4) for a, b in zip(tradeoff_summary["stage3_p95_step_s_mean"], tradeoff_summary["stage3_p95_step_s_std"])],
    "runtime / data-only": [pm(a, b, digits=2) for a, b in zip(tradeoff_summary["stage3_runtime_multiplier_vs_dataonly_mean"], tradeoff_summary["stage3_runtime_multiplier_vs_dataonly_std"])],
    "gain over data-only / extra s": [pm(a, b, digits=3) for a, b in zip(tradeoff_summary["gain_over_dataonly_pct_points_per_extra_second_mean"], tradeoff_summary["gain_over_dataonly_pct_points_per_extra_second_std"])],
    "nearest-train RMS": [pm(a, b, digits=4) for a, b in zip(tradeoff_summary["nearest_train_rms_mean_mean"], tradeoff_summary["nearest_train_rms_mean_std"])],
    "process peak-RSS delta (MB)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["process_peak_rss_delta_mb_mean"], tradeoff_summary["process_peak_rss_delta_mb_std"])],
    "reported device peak (MB)": [pm(a, b, digits=1) for a, b in zip(tradeoff_summary["device_peak_mb_mean"], tradeoff_summary["device_peak_mb_std"])],
})
print(tradeoff_display.to_string(index=False))


print_section("C. EMPIRICAL STAGE-III TIME COMPLEXITY AND SATURATION / PARETO DIAGNOSTICS")
runtime_rows = []
saturation_rows = []
pareto_rows = []
for ntraj, group in tradeoff_summary.groupby("n_trajectories"):
    g = group.sort_values("ambient_extra_points")
    x = g["ambient_extra_points"].to_numpy(dtype=float)
    y = g["stage3_mean_step_s_mean"].to_numpy(dtype=float)
    if len(x) >= 2 and np.all(np.isfinite(y)):
        slope, intercept = np.polyfit(x, y, 1)
        pred = intercept + slope * x
        ss_res = np.sum((y - pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan
    else:
        slope = intercept = r2 = np.nan
    runtime_rows.append({
        "Ntraj": int(ntraj),
        "intercept_seconds_per_step": intercept,
        "seconds_per_step_per_ambient_point": slope,
        "milliseconds_per_step_per_100_ambient_points": 1000.0 * 100.0 * slope,
        "R_squared": r2,
    })

    gains = g["combined_reduction_vs_stage1_pct_mean"].to_numpy(dtype=float)
    best_gain = float(np.nanmax(gains))
    target = 0.90 * max(best_gain, 0.0)
    qualifying = g[g["combined_reduction_vs_stage1_pct_mean"] >= target]
    sat = qualifying.iloc[0] if not qualifying.empty else g.iloc[-1]
    saturation_rows.append({
        "Ntraj": int(ntraj),
        "best_combined_gain_pct": best_gain,
        "smallest_total_reaching_90pct_best": int(sat["collocation_total"]),
        "ambient_extras_at_90pct": int(sat["ambient_extra_points"]),
        "stage3_time_at_90pct_s": float(sat["stage3_time_s_mean"]),
    })

    # Non-dominated points: maximize gain and minimize Stage-III time.
    for _, row in g.iterrows():
        dominated = False
        for _, other in g.iterrows():
            no_slower = other["stage3_time_s_mean"] <= row["stage3_time_s_mean"] + 1e-12
            no_worse = other["combined_reduction_vs_stage1_pct_mean"] >= row["combined_reduction_vs_stage1_pct_mean"] - 1e-12
            strictly = (
                other["stage3_time_s_mean"] < row["stage3_time_s_mean"] - 1e-12
                or other["combined_reduction_vs_stage1_pct_mean"] > row["combined_reduction_vs_stage1_pct_mean"] + 1e-12
            )
            if no_slower and no_worse and strictly:
                dominated = True
                break
        if not dominated:
            pareto_rows.append({
                "Ntraj": int(ntraj),
                "S_total": int(row["collocation_total"]),
                "ambient_extras": int(row["ambient_extra_points"]),
                "combined_gain_pct": float(row["combined_reduction_vs_stage1_pct_mean"]),
                "stage3_time_s": float(row["stage3_time_s_mean"]),
            })

runtime_scaling = pd.DataFrame(runtime_rows)
saturation = pd.DataFrame(saturation_rows)
pareto = pd.DataFrame(pareto_rows)
print("Empirical linear fit: steady seconds/step = intercept + slope × ambient-extra-count")
print(runtime_scaling.to_string(index=False, float_format=lambda x: f"{x:.6g}"))
print("\nSmallest budget reaching at least 90% of the best observed combined gain")
print(saturation.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nEmpirical Pareto-efficient accuracy/runtime points")
print(pareto.to_string(index=False, float_format=lambda x: f"{x:.4g}"))


# -----------------------------------------------------------------------------
# Unified E10 stage/objective/library/m/weight ablations at reference S=384.
# -----------------------------------------------------------------------------
print_section("D. STAGE PROGRESSION AND CONTROLS AT THE REFERENCE COLLOCATION BUDGET")
stage_rows = enriched[enriched["family"] == "stage"].copy()
stage_order = ["stage1_baseline", "stage2_frozen", "stage3_data_only", "stage3_random_W", "stage3_full"]
stage_summary = mean_std_frame(
    stage_rows,
    ["n_trajectories", "variant", "label"],
    [
        "drift_mse", "sigma_mse", "val_nll", "test_nll",
        "combined_reduction_vs_stage1_pct", "stage2_det", "stage3_det",
        "stage2_param_delta_max", "stage2_time_s", "stage3_time_s",
    ],
)
stage_summary["order"] = stage_summary["variant"].map({v: i for i, v in enumerate(stage_order)})
stage_summary = stage_summary.sort_values(["n_trajectories", "order"], ascending=[False, True])
stage_display = pd.DataFrame({
    "Ntraj": stage_summary["n_trajectories"],
    "Stage/control": stage_summary["label"],
    "Drift MSE": [pm(a, b, True) for a, b in zip(stage_summary["drift_mse_mean"], stage_summary["drift_mse_std"])],
    "Diffusion MSE": [pm(a, b, True) for a, b in zip(stage_summary["sigma_mse_mean"], stage_summary["sigma_mse_std"])],
    "Combined gain vs Stage I (%)": [pm(a, b, digits=1) for a, b in zip(stage_summary["combined_reduction_vs_stage1_pct_mean"], stage_summary["combined_reduction_vs_stage1_pct_std"])],
    "Stage-II held-out det": [pm(a, b, True) for a, b in zip(stage_summary["stage2_det_mean"], stage_summary["stage2_det_std"])],
    "Stage-III held-out det": [pm(a, b, True) for a, b in zip(stage_summary["stage3_det_mean"], stage_summary["stage3_det_std"])],
    "max |Δtheta| in Stage II": [pm(a, b, True) for a, b in zip(stage_summary["stage2_param_delta_max_mean"], stage_summary["stage2_param_delta_max_std"])],
    "Stage-II time (s)": [pm(a, b, digits=1) for a, b in zip(stage_summary["stage2_time_s_mean"], stage_summary["stage2_time_s_std"])],
    "Stage-III time (s)": [pm(a, b, digits=1) for a, b in zip(stage_summary["stage3_time_s_mean"], stage_summary["stage3_time_s_std"])],
})
print(stage_display.to_string(index=False))


def print_ablation_family(family, title, group_cols, sort_cols=None):
    print_section(title)
    df = enriched[enriched["family"] == family].copy()
    if df.empty:
        print("No rows were requested for this family.")
        return pd.DataFrame()
    summary = mean_std_frame(
        df,
        group_cols,
        [
            "drift_mse", "sigma_mse", "val_nll", "test_nll",
            "combined_reduction_vs_stage1_pct", "stage2_det", "stage2_close",
            "gram_eff_rank", "stage2_time_s", "stage3_time_s",
        ],
    )
    if sort_cols:
        summary = summary.sort_values(sort_cols)
    cols = group_cols + [
        "n_seeds", "drift_mse_mean", "sigma_mse_mean", "val_nll_mean",
        "combined_reduction_vs_stage1_pct_mean", "stage2_det_mean",
        "stage2_close_mean", "gram_eff_rank_mean", "stage2_time_s_mean", "stage3_time_s_mean",
    ]
    print(summary[cols].to_string(index=False, float_format=lambda x: f"{x:.6g}"))
    return summary


objective_summary = print_ablation_family(
    "objective",
    "E. DISCOVERY-OBJECTIVE ABLATION",
    ["n_trajectories", "objective_mode", "label"],
    ["n_trajectories", "objective_mode"],
)

library_summary = print_ablation_family(
    "library",
    "F. REDUCED VERSUS FULL 20D LIBRARY",
    ["n_trajectories", "library", "R", "label"],
    ["n_trajectories", "R"],
)

m_summary = print_ablation_family(
    "m_sweep",
    "G. NUMBER OF LEARNED GENERATORS m",
    ["n_trajectories", "m", "R"],
    ["n_trajectories", "m"],
)

print("\nValidation-NLL selection of m and comparison with the coefficient-error plateau")
m_selection_rows = []
if not m_summary.empty:
    for ntraj, group in m_summary.groupby("n_trajectories"):
        g = group.sort_values("m")
        selected = g.loc[g["val_nll_mean"].idxmin()]
        best_error = float(np.min(1.0 - g["combined_reduction_vs_stage1_pct_mean"] / 100.0))
        rel_error = 1.0 - g["combined_reduction_vs_stage1_pct_mean"] / 100.0
        plateau_limit = best_error * (1.0 + E10_CONFIG["M_PLATEAU_RELATIVE_TOLERANCE"])
        plateau_m = g.loc[rel_error <= plateau_limit, "m"].astype(int).tolist()
        m_selection_rows.append({
            "Ntraj": int(ntraj),
            "validation_selected_m": int(selected["m"]),
            "selected_val_nll": float(selected["val_nll_mean"]),
            "coefficient_plateau_m_values": plateau_m,
            "selection_on_plateau": int(selected["m"]) in plateau_m,
        })
m_selection = pd.DataFrame(m_selection_rows)
print(m_selection.to_string(index=False))


weight_summary = print_ablation_family(
    "weight_sweep",
    "H. ONE-AT-A-TIME 0.1× / 1× / 10× LOSS-WEIGHT SENSITIVITY",
    ["n_trajectories", "weight_name", "weight_multiplier", "lambda_det", "lambda_close", "lambda_ind", "lambda_norm", "lambda_gl"],
    ["n_trajectories", "weight_name", "weight_multiplier"],
)


# -----------------------------------------------------------------------------
# Conservative automated checks. These summarize results; they do not replace
# scientific interpretation.
# -----------------------------------------------------------------------------
print_section("I. AUTOMATED OPERATIONAL CHECKS")
checks = {}
for ntraj, group in tradeoff_summary.groupby("n_trajectories"):
    g = group.sort_values("collocation_total")
    small = g.iloc[0]
    large = g.iloc[-1]
    checks[f"ntraj_{int(ntraj)}_largest_beats_smallest_combined"] = bool(
        large["combined_reduction_vs_stage1_pct_mean"] > small["combined_reduction_vs_stage1_pct_mean"]
    )
    checks[f"ntraj_{int(ntraj)}_largest_beats_vanilla_drift"] = bool(
        large["drift_reduction_vs_stage1_pct_mean"] > 0.0
    )
    checks[f"ntraj_{int(ntraj)}_largest_beats_vanilla_diffusion"] = bool(
        large["sigma_reduction_vs_stage1_pct_mean"] > 0.0
    )
    checks[f"ntraj_{int(ntraj)}_runtime_increases_with_budget"] = bool(
        large["stage3_mean_step_s_mean"] > small["stage3_mean_step_s_mean"]
    )

for ntraj in E10_CONFIG["TRAJECTORY_COUNTS"]:
    s = enriched[(enriched["family"] == "stage") & (enriched["n_trajectories"] == int(ntraj))]
    full = s[s["variant"] == "stage3_full"]
    rand = s[s["variant"] == "stage3_random_W"]
    data = s[s["variant"] == "stage3_data_only"]
    if not full.empty and not rand.empty:
        checks[f"ntraj_{int(ntraj)}_learned_W_beats_random_W"] = bool(
            full["combined_relative_error_vs_stage1"].mean() < rand["combined_relative_error_vs_stage1"].mean()
        )
    if not full.empty and not data.empty:
        checks[f"ntraj_{int(ntraj)}_learned_W_beats_data_only"] = bool(
            full["combined_relative_error_vs_stage1"].mean() < data["combined_relative_error_vs_stage1"].mean()
        )

for name, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {name}")
print("Overall operational verdict:", "PASS" if checks and all(checks.values()) else "MIXED / INSPECT TABLES")


# -----------------------------------------------------------------------------
# Exports.
# -----------------------------------------------------------------------------
raw_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_raw_per_run.csv")
enriched_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_enriched_per_run.csv")
tradeoff_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_collocation_tradeoff_summary.csv")
runtime_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_runtime_scaling.csv")
saturation_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_saturation_summary.csv")
pareto_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_pareto_points.csv")
stage_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_stage_summary.csv")
objective_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_objective_summary.csv")
library_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_library_summary.csv")
m_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_m_summary.csv")
weight_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_weight_summary.csv")
verdict_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_operational_checks.json")
metadata_path = os.path.join(E10_CONFIG["OUTPUT_DIR"], "E10_20D_metadata.json")

raw.to_csv(raw_path, index=False)
enriched.to_csv(enriched_path, index=False)
tradeoff_summary.to_csv(tradeoff_path, index=False)
runtime_scaling.to_csv(runtime_path, index=False)
saturation.to_csv(saturation_path, index=False)
pareto.to_csv(pareto_path, index=False)
stage_summary.to_csv(stage_path, index=False)
objective_summary.to_csv(objective_path, index=False)
library_summary.to_csv(library_path, index=False)
m_summary.to_csv(m_path, index=False)
weight_summary.to_csv(weight_path, index=False)
with open(verdict_path, "w") as f:
    json.dump({"checks": checks, "all_pass": bool(checks and all(checks.values()))}, f, indent=2)
with open(metadata_path, "w") as f:
    json.dump(E10_METADATA, f, indent=2)

print_section("EXPORTED FILES")
for path in [
    raw_path, enriched_path, tradeoff_path, runtime_path, saturation_path,
    pareto_path, stage_path, objective_path, library_path, m_path, weight_path,
    verdict_path, metadata_path,
]:
    print(path)

